# Garbage Bag Detection: YOLOv11m vs Gemma 4 31B-QAT

## 1. Controles de execução

In [ ]:
# Runtime local: imports tolerantes, .env, seed, helpers e Matplotlib.
import os

os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("MPLCONFIGDIR", str((__import__("pathlib").Path(".cache/matplotlib")).resolve()))

import base64
import gc
import hashlib
import importlib.metadata as importlib_metadata
import io
import json
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
from collections import defaultdict
from pathlib import Path

Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm import tqdm

try:
    import yaml
except Exception as e:
    yaml = None
    print(f"PyYAML indisponivel nesta sessao: {e}")

try:
    from dotenv import find_dotenv, load_dotenv
    DOTENV_PATH = find_dotenv(filename=".env", usecwd=True)
    load_dotenv(DOTENV_PATH, override=False)
except Exception as e:
    DOTENV_PATH = ""
    print(f"python-dotenv indisponivel nesta sessao: {e}")

try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except Exception:
    pillow_heif = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except Exception as e:
    torch = None
    print(f"Torch indisponivel nesta sessao: {e}")

try:
    import cv2
except Exception as e:
    cv2 = None
    print(f"OpenCV indisponivel nesta sessao: {e}")

EXTS_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".heic"}
EXTS_MANIFESTO_DADOS = EXTS_IMG | {".txt", ".yaml", ".csv", ".json"}


def listar_imagens(pasta) -> list:
    pasta = Path(pasta)
    if not pasta.exists():
        return []
    return sorted(p for p in pasta.rglob("*") if p.is_file() and p.suffix.lower() in EXTS_IMG)


def contar_imagens(pasta) -> int:
    return len(listar_imagens(pasta))


def contar_arquivos_json(path):
    path = Path(path)
    return len(list(path.glob("*.json"))) if path.exists() else 0


def limpar_diretorio(pasta):
    pasta = Path(pasta)
    if pasta.exists():
        shutil.rmtree(pasta)
    pasta.mkdir(parents=True, exist_ok=True)


def ler_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def salvar_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    tmp.replace(path)


def env_bool(name, default=False):
    raw = os.getenv(name)
    if raw is None:
        return bool(default)
    return raw.strip().casefold() in {"1", "true", "yes", "y", "sim", "on"}


def pacote_versao(distribuicao: str):
    try:
        return importlib_metadata.version(distribuicao)
    except importlib_metadata.PackageNotFoundError:
        return None


def versoes_pacotes():
    pacotes = {
        "numpy": "numpy", "pandas": "pandas", "torch": "torch",
        "ultralytics": "ultralytics", "roboflow": "roboflow",
        "openai": "openai", "opencv_python": "opencv-python",
        "Pillow": "Pillow", "pillow_heif": "pillow-heif",
        "imagehash": "ImageHash", "jsonschema": "jsonschema",
        "scikit_learn": "scikit-learn", "statsmodels": "statsmodels",
        "matplotlib": "matplotlib", "PyYAML": "PyYAML",
    }
    return {nome: pacote_versao(dist) for nome, dist in pacotes.items()}


def git_commit(repo_dir):
    repo_dir = Path(repo_dir)
    if not (repo_dir / ".git").exists():
        return None
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
    except Exception:
        return None


def sha256_arquivo(path, bloco=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(bloco), b""):
            h.update(chunk)
    return h.hexdigest()


def hash_manifesto_diretorio(root, extensoes=None):
    root = Path(root)
    if not root.exists():
        return {"existe": False, "n_arquivos": 0, "sha256_manifesto": None}
    linhas = []
    for p in sorted(x for x in root.rglob("*") if x.is_file()):
        if extensoes is not None and p.suffix.lower() not in extensoes:
            continue
        linhas.append(f"{p.relative_to(root).as_posix()}\t{p.stat().st_size}\t{sha256_arquivo(p)}")
    payload = "\n".join(linhas).encode("utf-8")
    return {"existe": True, "n_arquivos": len(linhas),
            "sha256_manifesto": hashlib.sha256(payload).hexdigest()}


def imagem_para_data_url_jpeg(path, quality=95):
    with Image.open(path) as im:
        rgb = ImageOps.exif_transpose(im).convert("RGB")
        buf = io.BytesIO()
        rgb.save(buf, format="JPEG", quality=quality)
    payload = buf.getvalue()
    b64 = base64.b64encode(payload).decode()
    return {
        "data_url": f"data:image/jpeg;base64,{b64}",
        "mime_type": "image/jpeg",
        "payload_bytes": len(payload),
        "payload_sha256": hashlib.sha256(payload).hexdigest(),
    }

print(f"Python {sys.version.split()[0]} | cwd={Path.cwd()}")


In [ ]:
# Run All reiniciavel. Cada flag pode ser sobrescrita no .env.
RUN_ALL_PIPELINE = env_bool("RUN_ALL_PIPELINE", True)
TRAIN_ONLY_PIPELINE = env_bool("TRAIN_ONLY_PIPELINE", False)
RUN_DOWNLOAD_DADOS = env_bool("RUN_DOWNLOAD_DADOS", RUN_ALL_PIPELINE)
RUN_DOWNLOAD_TREINO = env_bool("RUN_DOWNLOAD_TREINO", RUN_DOWNLOAD_DADOS)
RUN_DOWNLOAD_TESTE = env_bool("RUN_DOWNLOAD_TESTE", RUN_DOWNLOAD_DADOS and not TRAIN_ONLY_PIPELINE)
RUN_RECONSTRUIR_UNIFIED = env_bool("RUN_RECONSTRUIR_UNIFIED", RUN_ALL_PIPELINE)
RUN_RECONSTRUIR_TESTE = env_bool("RUN_RECONSTRUIR_TESTE", RUN_ALL_PIPELINE and not TRAIN_ONLY_PIPELINE)
RUN_TREINO_YOLO = env_bool("RUN_TREINO_YOLO", RUN_ALL_PIPELINE)
RUN_YOLO_INFERENCIA = env_bool("RUN_YOLO_INFERENCIA", RUN_ALL_PIPELINE and not TRAIN_ONLY_PIPELINE)
RUN_VLM_CLASSIFICACAO = env_bool("RUN_VLM_CLASSIFICACAO", RUN_ALL_PIPELINE and not TRAIN_ONLY_PIPELINE)
RUN_VLM_LOCALIZACAO = env_bool("RUN_VLM_LOCALIZACAO", RUN_ALL_PIPELINE and not TRAIN_ONLY_PIPELINE)
RUN_AUDITORIA_PHASH = env_bool("RUN_AUDITORIA_PHASH", RUN_ALL_PIPELINE and not TRAIN_ONLY_PIPELINE)

FORCE_DOWNLOAD_DADOS = env_bool("FORCE_DOWNLOAD_DADOS", False)
FORCE_DOWNLOAD_TREINO = env_bool("FORCE_DOWNLOAD_TREINO", FORCE_DOWNLOAD_DADOS)
FORCE_DOWNLOAD_TESTE = env_bool("FORCE_DOWNLOAD_TESTE", FORCE_DOWNLOAD_DADOS)
FORCE_REBUILD_DADOS = env_bool("FORCE_REBUILD_DADOS", False)
FORCE_TREINO_YOLO = env_bool("FORCE_TREINO_YOLO", False)
FORCE_YOLO_INFERENCIA = env_bool("FORCE_YOLO_INFERENCIA", False)

VLM_LOC_ESTRATEGIA = os.getenv("VLM_LOC_ESTRATEGIA", "todos")
VLM_LOC_MAX_IMAGES = None
VLM_LOC_MAX_DETECCOES = 20

CACHE_VALIDACAO_ESTRITA = env_bool("CACHE_VALIDACAO_ESTRITA", True)
CACHE_CAMPOS_IGNORADOS = {"lm_studio_base_url"}

VLM_CLASS_MAX_TOKENS = 300
VLM_LOC_MAX_TOKENS = 1200
VLM_TIMEOUT_CLASS = 45
VLM_TIMEOUT_LOC = 90
VLM_MAX_TENTATIVAS = 2
VLM_JPEG_QUALITY = 95

MODELO_VLM_ALVO = os.getenv("LM_STUDIO_MODEL", "google/gemma-4-31b-qat")
MODELO_VLM_REPOSITORIO_BASE = "lmstudio-community/gemma-4-31B-it-QAT-GGUF"
MODELO_VLM_ARQUIVO = "gemma-4-31B-it-QAT-Q4_0.gguf"
MODELO_VLM_FORMATO = "GGUF"
MODELO_VLM_QUANTIZACAO = "Q4_0"
MODELO_VLM_ARQUITETURA = "gemma4"
MODELO_VLM_TAMANHO_DISCO_GB = 18.85
LM_STUDIO_APP_VERSION = "0.4.15 (Build 2)"

PIPELINE_STATE_PATH = Path("outputs/pipeline_state.json")
PIPELINE_STATE_SCHEMA = 2
pipeline_state = ler_json(PIPELINE_STATE_PATH, default={}) or {}
if pipeline_state.get("schema_version") != PIPELINE_STATE_SCHEMA:
    pipeline_state = {"schema_version": PIPELINE_STATE_SCHEMA, "steps": {}}


def pipeline_fingerprint(payload) -> str:
    serializado = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(serializado).hexdigest()[:16]


def _artefato_pronto(path) -> bool:
    path = Path(path)
    if not path.exists():
        return False
    return not path.is_dir() or any(path.rglob("*"))


def etapa_concluida(nome, fingerprint, artefatos=()) -> bool:
    etapa = pipeline_state.get("steps", {}).get(nome, {})
    return (etapa.get("status") == "completed" and etapa.get("fingerprint") == fingerprint
            and all(_artefato_pronto(path) for path in artefatos))


def marcar_etapa(nome, status, fingerprint, **detalhes):
    pipeline_state.setdefault("steps", {})[nome] = {
        "status": status, "fingerprint": fingerprint,
        "updated_at": pd.Timestamp.now().isoformat(), **detalhes,
    }
    salvar_json(pipeline_state, PIPELINE_STATE_PATH)


def executar_etapa(nome, fingerprint, acao, artefatos=(), force=False):
    if not force and etapa_concluida(nome, fingerprint, artefatos):
        print(f"[resume] etapa {nome!r} ja concluida; reutilizando artefatos.")
        return None
    marcar_etapa(nome, "running", fingerprint)
    try:
        resultado = acao()
        marcar_etapa(nome, "completed", fingerprint, artefatos=[str(x) for x in artefatos])
        return resultado
    except BaseException as exc:
        marcar_etapa(nome, "failed", fingerprint, error=repr(exc))
        raise


In [ ]:
EXTS_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".heic"}
EXTS_MANIFESTO_DADOS = EXTS_IMG | {".txt", ".yaml", ".csv"}

TESTE_IMG = Path("data/teste/images")
TESTE_LBL = Path("data/teste/labels")
OUTPUTS_DIR = Path("outputs")
FIG_DIR = OUTPUTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PROJECT = Path("runs/detect/models").resolve()
YOLO_RUN_NAME = os.getenv("YOLO_RUN_NAME", "yolov11m_sacos_lixo_roboflow_grouped_split")
TRAIN_DIR = TRAIN_PROJECT / YOLO_RUN_NAME
BEST_PT = TRAIN_DIR / "weights/best.pt"
LAST_PT = TRAIN_DIR / "weights/last.pt"
YOLO_CACHE_DIR = Path("outputs/yolo_cache") / YOLO_RUN_NAME
YOLO_CACHE_DIR.mkdir(parents=True, exist_ok=True)
YOLO_EPOCHS = int(os.getenv("YOLO_EPOCHS", "100"))
PROMPT_CLASSIFICACAO_PATH = Path("prompt_gemma_classificacao.txt")
PROMPT_LOCALIZACAO_PATH = Path("prompt_gemma_localizacao.txt")
DATA_YAML = Path("data/unified/data.yaml")


In [ ]:
# data.yaml
DATA_YAML = Path("data/unified/data.yaml")
if RUN_RECONSTRUIR_UNIFIED or not DATA_YAML.exists():
    DATA_YAML.parent.mkdir(parents=True, exist_ok=True)
    payload_yaml = (
        "path: data/unified\n"
        "train: images/train\n"
        "val: images/val\n"
        "nc: 1\n"
        "names:\n"
        "  0: saco_de_lixo\n"
    )
    DATA_YAML.write_text(payload_yaml, encoding="utf-8")
    print(f"data.yaml atualizado em {DATA_YAML}")
else:
    print(f"data.yaml existente preservado: {DATA_YAML}")
print(DATA_YAML.read_text(encoding="utf-8") if DATA_YAML.exists() else "data.yaml ausente")


In [ ]:
# Configuracao fixa do experimento; o .env pode alterar somente workspace/projeto/versao.
DATASET_CONFIG = [
    {
        "key": "GARBAGE_8UZHA",
        "workspace": os.getenv("ROBOFLOW_WORKSPACE_GARBAGE_8UZHA", "project-r1emy"),
        "project": os.getenv("ROBOFLOW_PROJECT_GARBAGE_8UZHA", "garbage-8uzha"),
        "version": int(os.getenv("ROBOFLOW_VERSION_GARBAGE_8UZHA", "1")),
        "positive_classes": ["black_bag", "white_bag"],
        "expected_positive_images": 1838,
    },
    {
        "key": "GARBAGE_MVZG3",
        "workspace": os.getenv("ROBOFLOW_WORKSPACE_GARBAGE_MVZG3", "bill-lhxqf"),
        "project": os.getenv("ROBOFLOW_PROJECT_GARBAGE_MVZG3", "garbage-mvzg3"),
        "version": int(os.getenv("ROBOFLOW_VERSION_GARBAGE_MVZG3", "1")),
        "positive_classes": ["trash bag"],
        "background_classes": ["Roadway"],
        "expected_positive_images": 865,
    },
]
DATASET_POSITIVOS = [item["key"] for item in DATASET_CONFIG]
BACKGROUND_CONFIG = {
    "class": "Roadway",
    "ratio_over_positive_images": float(os.getenv("UNIFIED_BACKGROUND_RATIO", "0.10")),
    "max_images": int(os.getenv("UNIFIED_BACKGROUND_MAX_IMAGES", "271")),
}
TEST_DATASET_CONFIG = {
    "workspace": os.getenv("ROBOFLOW_WORKSPACE_TESTE", "jaime-teixeira"),
    "project": os.getenv("ROBOFLOW_PROJECT_TESTE", "urban-waste-brazil"),
    "version": int(os.getenv("ROBOFLOW_VERSION_TESTE", "1")),
}
PREPARE_SCRIPT_SHA256 = sha256_arquivo("scripts/prepare_datasets.py")
TRAIN_PREP_VERSION = "bags-roadway-v1"
TEST_PREP_VERSION = "urban-waste-brazil-v1"
TRAIN_DATA_CONFIG_FINGERPRINT = pipeline_fingerprint({
    "train": DATASET_CONFIG, "background": BACKGROUND_CONFIG,
    "prepare_version": TRAIN_PREP_VERSION,
})
TEST_DATA_CONFIG_FINGERPRINT = pipeline_fingerprint({
    "test": TEST_DATASET_CONFIG, "prepare_version": TEST_PREP_VERSION,
})
DATA_CONFIG_FINGERPRINT = pipeline_fingerprint({
    "train": TRAIN_DATA_CONFIG_FINGERPRINT, "test": TEST_DATA_CONFIG_FINGERPRINT,
})

for d in [
    "data/teste/labels", "models", "outputs/vlm_cache", "outputs/vlm_loc_cache",
    "outputs/yolo_cache", "outputs/figures", "outputs/gemma-4-31b-qat/analises",
]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Datasets de treino configurados:")
for item in DATASET_CONFIG:
    print(f"  - {item['workspace']}/{item['project']} v{item['version']} | positivas={item['positive_classes']}")
print(f"  - negativos: garbage-mvzg3/{BACKGROUND_CONFIG['class']} | "
      f"{BACKGROUND_CONFIG['ratio_over_positive_images']:.0%} das positivas, max={BACKGROUND_CONFIG['max_images']}")
print(f"Teste: {TEST_DATASET_CONFIG['workspace']}/{TEST_DATASET_CONFIG['project']} v{TEST_DATASET_CONFIG['version']}")
print(f"Fingerprints | treino={TRAIN_DATA_CONFIG_FINGERPRINT} | teste={TEST_DATA_CONFIG_FINGERPRINT}")


## 2. Download e preparo dos datasets


In [ ]:
# Download antecipado somente dos datasets de treino.
DOWNLOAD_TRAIN_ARTIFACTS = [Path("data") / f"raw_{item['project']}" for item in DATASET_CONFIG]


def baixar_datasets_treino():
    args = [sys.executable, "scripts/prepare_datasets.py", "--download-train"]
    if FORCE_DOWNLOAD_TREINO:
        args.append("--overwrite")
    subprocess.run(args, check=True)


if RUN_DOWNLOAD_TREINO:
    executar_etapa("download_train_datasets", TRAIN_DATA_CONFIG_FINGERPRINT, baixar_datasets_treino,
                   artefatos=DOWNLOAD_TRAIN_ARTIFACTS, force=FORCE_DOWNLOAD_TREINO)
else:
    print("Download dos datasets de treino desativado; dados locais serao reutilizados.")


In [ ]:
# Reconstrucao antecipada somente do conjunto unificado de treino/validacao.
if RUN_RECONSTRUIR_UNIFIED:
    executar_etapa(
        "rebuild_unified", TRAIN_DATA_CONFIG_FINGERPRINT,
        lambda: subprocess.run([sys.executable, "scripts/prepare_datasets.py", "--rebuild"], check=True),
        artefatos=["data/unified/data.yaml", "data/unified/manifest.csv", "data/unified/summary.json"],
        force=FORCE_REBUILD_DADOS,
    )
else:
    print("Reconstrucao do treino desativada; artefatos existentes serao reutilizados.")


In [ ]:
def parse_json_robusto(raw: str) -> dict:
    if not raw:
        return {"parse_error": True}
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        i, j = raw.find("{"), raw.rfind("}") + 1
        if 0 <= i < j:
            try:
                return json.loads(raw[i:j])
            except json.JSONDecodeError:
                pass
    return {"parse_error": True}



def cache_valido(p: Path, esperado: dict, strict=CACHE_VALIDACAO_ESTRITA) -> bool:
    try:
        salvo = ler_json(p)
        if not isinstance(salvo, dict):
            return False
        if salvo.get("erro") or salvo.get("parse_error"):
            return False
        ignorar = set() if strict else CACHE_CAMPOS_IGNORADOS
        return all(salvo.get(k) == v for k, v in esperado.items() if k not in ignorar)
    except Exception:
        return False


## 3. Leitura do teste disponível em cache


In [ ]:
def carregar_conjunto_teste():
    images = listar_imagens(TESTE_IMG)
    image_by_id = {p.name: p for p in images}
    gt_path = Path("data/teste/ground_truth.csv")
    if gt_path.exists():
        gt = pd.read_csv(gt_path)
    elif images:
        registros = []
        for img_path in images:
            lbl = TESTE_LBL / (img_path.stem + ".txt")
            n_bboxes = len([line for line in lbl.read_text().splitlines() if line.strip()]) if lbl.exists() else 0
            registros.append({"image_id": img_path.name, "tem_lixo": int(n_bboxes > 0), "n_bboxes": n_bboxes})
        gt = pd.DataFrame(registros)
        gt_path.parent.mkdir(parents=True, exist_ok=True)
        gt.to_csv(gt_path, index=False)
    else:
        gt = pd.DataFrame(columns=["image_id", "tem_lixo", "n_bboxes"])
    if not gt.empty:
        gt["tem_lixo"] = gt["tem_lixo"].astype(int)
        gt["n_bboxes"] = gt["n_bboxes"].fillna(0).astype(int)
    image_ids = gt["image_id"].astype(str).tolist() if "image_id" in gt.columns else [p.name for p in images]
    com_saco = int(gt["tem_lixo"].sum()) if "tem_lixo" in gt.columns else 0
    sem_saco = len(gt) - com_saco
    gt_by_id = dict(zip(gt["image_id"], gt["tem_lixo"])) if "image_id" in gt.columns else {}
    return images, image_by_id, gt, image_ids, com_saco, sem_saco, gt_by_id


test_images, test_image_by_id, df_gt, test_image_ids, n_com, n_sem, gt_map = carregar_conjunto_teste()
print(f"Teste disponivel antes do treino: {len(test_images)} imagens locais; {len(test_image_ids)} IDs no GT em cache.")


In [ ]:
print("Artefatos principais:")
for p in [
    BEST_PT,
    Path("outputs/predictions_yolo.json"),
    Path("outputs/gemma-4-31b-qat/predictions_vlm.json"),
    Path("outputs/gemma-4-31b-qat/predictions_vlm_localizacao.json"),
]:
    print(f"  {'ok' if p.exists() else '--'} {p}")


## 4. YOLO


In [ ]:
model_yolo = None
GPU_NAME = "CPU/indisponivel"

if torch is not None and torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)

YOLO_TRAIN_CONFIG = {
    "base_weights": os.getenv("YOLO_TRAIN_INIT_WEIGHTS", os.getenv("YOLO_PRETRAINED", "yolo11m.pt")),
    "epochs": YOLO_EPOCHS, "batch": 12, "imgsz": 640, "optimizer": "AdamW",
    "lr0": 0.001, "patience": 20, "mosaic": 1.0, "fliplr": 0.5,
    "seed": SEED, "dataset_fingerprint": TRAIN_DATA_CONFIG_FINGERPRINT,
}
YOLO_TRAIN_FINGERPRINT = pipeline_fingerprint(YOLO_TRAIN_CONFIG)


def executar_treino_yolo_reiniciavel():
    if FORCE_TREINO_YOLO and TRAIN_DIR.exists():
        shutil.rmtree(TRAIN_DIR)
    train_config_path = TRAIN_DIR / "train_config.json"
    config_checkpoint = ler_json(train_config_path, default={}) or {}
    if LAST_PT.exists() and config_checkpoint.get("train_fingerprint") == YOLO_TRAIN_FINGERPRINT:
        print(f"[resume] retomando treinamento YOLO de {LAST_PT}")
        YOLO(str(LAST_PT)).train(resume=True)
    elif LAST_PT.exists():
        raise RuntimeError("last.pt pertence a outra configuracao. Use FORCE_TREINO_YOLO=true ou altere YOLO_RUN_NAME.")
    else:
        init_weights = YOLO_TRAIN_CONFIG["base_weights"]
        print(f"Iniciando treinamento YOLO em {TRAIN_DIR} a partir de {init_weights}")
        TRAIN_DIR.mkdir(parents=True, exist_ok=True)
        salvar_json({"train_fingerprint": YOLO_TRAIN_FINGERPRINT, "train_config": YOLO_TRAIN_CONFIG}, train_config_path)
        YOLO(init_weights).train(
            data=str(DATA_YAML.resolve()), epochs=YOLO_EPOCHS, batch=12, imgsz=640,
            optimizer="AdamW", lr0=0.001, patience=20, mosaic=1.0,
            fliplr=0.5, flipud=0.0, seed=SEED, deterministic=True, save=True,
            project=str(TRAIN_PROJECT), name=YOLO_RUN_NAME, exist_ok=True,
            device=0 if torch is not None and torch.cuda.is_available() else "cpu", workers=4, verbose=True,
        )
    if not BEST_PT.exists():
        raise FileNotFoundError(f"Treinamento encerrado sem best.pt: {BEST_PT}")
    run_provenance = {
        "train_fingerprint": YOLO_TRAIN_FINGERPRINT, "train_config": YOLO_TRAIN_CONFIG,
        "yolo_run_name": YOLO_RUN_NAME, "yolo_run_dir": str(TRAIN_DIR),
        "weights_path": str(BEST_PT), "weights_sha256": sha256_arquivo(BEST_PT),
        "data_yaml": str(DATA_YAML),
        "unified_summary": ler_json("data/unified/summary.json", default={}),
        "unified_split_audit": ler_json("data/unified/split_audit.json", default={}),
        "unified_manifest": hash_manifesto_diretorio("data/unified", EXTS_MANIFESTO_DADOS),
        "seed": SEED,
    }
    salvar_json(run_provenance, TRAIN_DIR / "run_provenance.json")

if RUN_TREINO_YOLO or RUN_YOLO_INFERENCIA:
    from ultralytics import YOLO

    if RUN_TREINO_YOLO:
        assert DATA_YAML.exists(), f"data.yaml nao encontrado: {DATA_YAML}"
        prov_existente = ler_json(TRAIN_DIR / "run_provenance.json", default={}) or {}
        if BEST_PT.exists() and prov_existente.get("train_fingerprint") == YOLO_TRAIN_FINGERPRINT and not FORCE_TREINO_YOLO:
            marcar_etapa("train_yolo", "completed", YOLO_TRAIN_FINGERPRINT, recovered_from_provenance=True)
            print(f"[resume] treinamento YOLO ja concluido: {BEST_PT}")
        else:
            executar_etapa("train_yolo", YOLO_TRAIN_FINGERPRINT, executar_treino_yolo_reiniciavel,
                           artefatos=[BEST_PT], force=FORCE_TREINO_YOLO)
        model_yolo = YOLO(str(BEST_PT))
        print(f"Pesos YOLO carregados: {BEST_PT}")
    elif BEST_PT.exists():
        print(f"Reutilizando pesos: {BEST_PT}")
        model_yolo = YOLO(str(BEST_PT))
    else:
        raise FileNotFoundError(f"Pesos YOLO nao encontrados: {BEST_PT}. Defina RUN_TREINO_YOLO=True ou ajuste YOLO_RUN_NAME.")
else:
    print("YOLO em modo cache: treino/inferencia desligados.")


### 4.1 Download e preparo do teste antes das inferências


In [ ]:
# Esta etapa ocorre depois do treinamento: indisponibilidade do teste nao descarta o progresso do YOLO.
DOWNLOAD_TEST_ARTIFACTS = [Path("data/raw_imagens_teste")]


def baixar_dataset_teste():
    args = [sys.executable, "scripts/prepare_datasets.py", "--download-test"]
    if FORCE_DOWNLOAD_TESTE:
        args.append("--overwrite")
    subprocess.run(args, check=True)


if RUN_DOWNLOAD_TESTE:
    executar_etapa("download_test_dataset", TEST_DATA_CONFIG_FINGERPRINT, baixar_dataset_teste,
                   artefatos=DOWNLOAD_TEST_ARTIFACTS, force=FORCE_DOWNLOAD_TESTE)
else:
    print("Download do dataset de teste desativado.")

if RUN_RECONSTRUIR_TESTE:
    executar_etapa(
        "rebuild_test", TEST_DATA_CONFIG_FINGERPRINT,
        lambda: subprocess.run([sys.executable, "scripts/prepare_datasets.py", "--rebuild-test"], check=True),
        artefatos=["data/teste/ground_truth.csv", "data/teste/summary.json"],
        force=FORCE_REBUILD_DADOS,
    )

test_images, test_image_by_id, df_gt, test_image_ids, n_com, n_sem, gt_map = carregar_conjunto_teste()
print(f"Teste pronto para inferencia: {len(test_images)} imagens | com saco={n_com} | sem saco={n_sem}")
if (RUN_YOLO_INFERENCIA or RUN_VLM_CLASSIFICACAO or RUN_VLM_LOCALIZACAO) and not test_images:
    raise RuntimeError("Treinamento preservado, mas o dataset de teste ainda nao esta disponivel. "
                       "Finalize o projeto Roboflow ou desative temporariamente as flags de inferencia.")


In [ ]:
def ler_imagem_bgr(path):
    if cv2 is None:
        raise RuntimeError("OpenCV indisponivel para inferencia YOLO.")
    with Image.open(path) as im:
        rgb = np.array(ImageOps.exif_transpose(im).convert("RGB"))
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


YOLO_WEIGHTS_SHA256 = sha256_arquivo(BEST_PT) if BEST_PT.exists() else None
YOLO_INFER_FINGERPRINT = pipeline_fingerprint({
    "weights_sha256": YOLO_WEIGHTS_SHA256, "conf": 0.05, "iou": 0.45,
    "test_dataset": TEST_DATASET_CONFIG,
})


def yolo_cache_path(img_path):
    img_path = Path(img_path)
    key = hashlib.sha256(img_path.name.encode("utf-8")).hexdigest()[:12]
    return YOLO_CACHE_DIR / f"{img_path.stem}_{key}.json"


def metadados_yolo_cache(img_path):
    img_path = Path(img_path)
    return {
        "inference_fingerprint": YOLO_INFER_FINGERPRINT,
        "weights_sha256": YOLO_WEIGHTS_SHA256,
        "image_id": img_path.name, "image_sha256": sha256_arquivo(img_path),
        "conf": 0.05, "iou": 0.45,
    }


def carregar_yolo_cache():
    preds = []
    for img_path in test_images:
        path = yolo_cache_path(img_path)
        meta = metadados_yolo_cache(img_path)
        if cache_valido(path, meta, strict=True):
            preds.append(ler_json(path))
    return preds


def executar_yolo_inferencia():
    assert model_yolo is not None, "Carregue model_yolo antes de executar YOLO."
    pendentes = list(test_images) if FORCE_YOLO_INFERENCIA else [
        img for img in test_images if not cache_valido(yolo_cache_path(img), metadados_yolo_cache(img), strict=True)
    ]
    concluidas = len(test_images) - len(pendentes)
    marcar_etapa("yolo_inference", "running", YOLO_INFER_FINGERPRINT,
                 completed=concluidas, total=len(test_images))
    try:
        for idx, img_path in enumerate(tqdm(pendentes, desc="YOLO"), start=1):
            t_e2e0 = time.perf_counter()
            img = ler_imagem_bgr(img_path)
            if torch is not None and torch.cuda.is_available():
                torch.cuda.synchronize()
            t_m0 = time.perf_counter()
            results = model_yolo.predict(img, conf=0.05, iou=0.45, verbose=False)
            if torch is not None and torch.cuda.is_available():
                torch.cuda.synchronize()
            t_model_manual = (time.perf_counter() - t_m0) * 1000
            t_e2e = (time.perf_counter() - t_e2e0) * 1000
            sp = results[0].speed
            t_infer = float(sp.get("preprocess", 0) + sp.get("inference", 0) + sp.get("postprocess", 0))
            resultado = {
                "deteccoes": [{"confidence": float(b.conf), "bbox_xywhn": b.xywhn[0].tolist()} for b in results[0].boxes],
                "latencia_inferencia_ms": round(t_infer, 2),
                "latencia_model_manual_ms": round(t_model_manual, 2),
                "latencia_end_to_end_ms": round(t_e2e, 2),
                **metadados_yolo_cache(img_path),
            }
            salvar_json(resultado, yolo_cache_path(img_path))
            if idx % 10 == 0 or idx == len(pendentes):
                marcar_etapa("yolo_inference", "running", YOLO_INFER_FINGERPRINT,
                             completed=concluidas + idx, total=len(test_images))
    except BaseException as exc:
        marcar_etapa("yolo_inference", "failed", YOLO_INFER_FINGERPRINT,
                     completed=len(carregar_yolo_cache()), total=len(test_images), error=repr(exc))
        raise

    yolo_preds = carregar_yolo_cache()
    payload = {
        "hardware": GPU_NAME,
        "yolo_run_name": YOLO_RUN_NAME,
        "weights_path": str(BEST_PT),
        "weights_sha256": sha256_arquivo(BEST_PT) if BEST_PT.exists() else None,
        "data_yaml": str(DATA_YAML),
        "unified_summary": ler_json("data/unified/summary.json", default={}),
        "unified_split_audit": ler_json("data/unified/split_audit.json", default={}),
        "unified_manifest": hash_manifesto_diretorio("data/unified", EXTS_MANIFESTO_DADOS),
        "teste_manifest": hash_manifesto_diretorio("data/teste", EXTS_MANIFESTO_DADOS),
        "predicoes": yolo_preds,
    }
    salvar_json(payload, "outputs/predictions_yolo.json")
    marcar_etapa("yolo_inference", "completed", YOLO_INFER_FINGERPRINT,
                 completed=len(yolo_preds), total=len(test_images))
    return yolo_preds


In [ ]:
yolo_payload = ler_json("outputs/predictions_yolo.json", default={})

if RUN_YOLO_INFERENCIA:
    yolo_preds = executar_yolo_inferencia()
elif test_images and BEST_PT.exists():
    yolo_preds = carregar_yolo_cache()
    print(f"YOLO carregado dos checkpoints por imagem: {len(yolo_preds)} predicoes")
elif yolo_payload.get("predicoes"):
    yolo_preds = yolo_payload["predicoes"]
    print(f"YOLO carregado do cache: {len(yolo_preds)} predicoes")
else:
    yolo_preds = []
    print("Sem predicoes YOLO carregadas.")

pred_by_id = {p["image_id"]: p for p in yolo_preds}
lat_y_infer = [p.get("latencia_inferencia_ms", 0) for p in yolo_preds if p.get("latencia_inferencia_ms")]
lat_y_e2e = [p.get("latencia_end_to_end_ms", 0) for p in yolo_preds if p.get("latencia_end_to_end_ms")]
print(f"YOLO disponivel: {len(pred_by_id)}/{len(test_images)} imagens")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


def imagem_tem_lixo(deteccoes, limiar):
    return int(any(d.get("confidence", 0.0) >= limiar for d in deteccoes))

ids_ok = [p["image_id"] for p in yolo_preds if p["image_id"] in gt_map]
LIMIARES = [0.25, 0.35, 0.50, 0.65]
linhas = []

for limiar in LIMIARES:
    y_true = [gt_map[i] for i in ids_ok]
    y_pred = [imagem_tem_lixo(pred_by_id[i]["deteccoes"], limiar) for i in ids_ok]
    if not y_true:
        continue
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    linhas.append({
        "limiar": limiar,
        "acuracia": round(accuracy_score(y_true, y_pred), 4),
        "precisao": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "especificidade": round(tn / (tn + fp), 4) if (tn + fp) else 0,
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "taxa_fp": round(fp / (fp + tn), 4) if (fp + tn) else 0,
        "fp": int(fp), "fn": int(fn), "tp": int(tp), "tn": int(tn),
    })

df_limiar = pd.DataFrame(linhas)
varredura_path = Path("outputs/varredura_limiares.csv")
if len(df_limiar):
    df_limiar.to_csv(varredura_path, index=False)
elif varredura_path.exists():
    df_limiar = pd.read_csv(varredura_path)

if len(df_limiar):
    print(df_limiar.to_string(index=False))
else:
    print("Sem dados para varredura de limiares YOLO.")


## 5. VLM: setup sem falhar se LM Studio estiver offline


In [ ]:
def _slug_modelo(model_id: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", model_id).strip("-").lower()


def _modelo_corresponde(model_id: str, esperado: str) -> bool:
    mid, exp = str(model_id).lower(), str(esperado).lower()
    return mid == exp or mid.startswith(exp) or mid.endswith(f"/{exp}") or exp in mid


def modelo_vlm_de_cache(slug):
    for d in [Path("outputs/vlm_cache") / slug, Path("outputs/vlm_loc_cache") / slug]:
        for f in sorted(d.glob("*.json"))[:5]:
            obj = ler_json(f, {})
            if obj.get("modelo_vlm"):
                return obj["modelo_vlm"]
    return None

LM_BASE_URL = os.getenv("LM_STUDIO_BASE_URL", "http://localhost:1234/v1")
MODELO_VLM_LABEL = "Gemma 4 31B-QAT"
VLM_RUN_SLUG = _slug_modelo(MODELO_VLM_ALVO.split("/")[-1])
OUTPUT_VLM_DIR = Path("outputs") / VLM_RUN_SLUG
VLM_CACHE_DIR = Path("outputs/vlm_cache") / VLM_RUN_SLUG
VLM_LOC_CACHE_DIR = Path("outputs/vlm_loc_cache") / VLM_RUN_SLUG
for d in [OUTPUT_VLM_DIR, VLM_CACHE_DIR, VLM_LOC_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

modelo_vlm = modelo_vlm_de_cache(VLM_RUN_SLUG) or MODELO_VLM_ALVO
client = None
LM_STUDIO_DISPONIVEL = False


In [ ]:
VLM_PRECISA_HTTP = RUN_VLM_CLASSIFICACAO or RUN_VLM_LOCALIZACAO

if VLM_PRECISA_HTTP:
    import requests
    from openai import OpenAI
    try:
        r = requests.get(f"{LM_BASE_URL}/models", timeout=5)
        r.raise_for_status()
        disponiveis = [m["id"] for m in r.json().get("data", [])]
        achado = next((m for m in disponiveis if _modelo_corresponde(m, MODELO_VLM_ALVO)), None)
        if achado is None:
            raise RuntimeError(f"Modelo {MODELO_VLM_ALVO!r} nao carregado. Disponiveis: {disponiveis}")
        modelo_vlm = achado
        client = OpenAI(base_url=LM_BASE_URL, api_key="lm-studio")
        LM_STUDIO_DISPONIVEL = True
    except Exception as e:
        print(f"LM Studio indisponivel para novas chamadas: {e}")
else:
    print("Chamadas VLM novas desligadas; usando caches se existirem.")

print(f"VLM slug={VLM_RUN_SLUG} | modelo_vlm={modelo_vlm} | cache_loc={VLM_LOC_CACHE_DIR}")


In [ ]:
def _suporta(**extra):
    if client is None:
        return None
    try:
        client.chat.completions.create(
            model=modelo_vlm,
            messages=[{"role": "user", "content": "ping"}],
            max_tokens=1,
            temperature=0,
            **extra,
        )
        return True
    except Exception:
        return False

VLM_SUPORTA_SEED = _suporta(seed=SEED)
if VLM_SUPORTA_SEED is None:
    VLM_SUPORTA_SEED = True  # compatibilidade com caches existentes

print(f"seed suportado/assumido: {VLM_SUPORTA_SEED}")


## 6. VLM classificação binária


In [ ]:
try:
    import jsonschema
except Exception as e:
    jsonschema = None
    print(f"jsonschema indisponivel; validacao local de schema sera pulada: {e}")

if not PROMPT_CLASSIFICACAO_PATH.exists():
    raise FileNotFoundError(f"Prompt de classificacao ausente: {PROMPT_CLASSIFICACAO_PATH}")
PROMPT_VLM = PROMPT_CLASSIFICACAO_PATH.read_text(encoding="utf-8").strip()

SCHEMA_VLM = {
    "type": "object", "required": ["tem_lixo"],
    "properties": {
        "tem_lixo": {"type": "boolean"},
        "descricao_breve": {"type": "string"},
        "confianca": {"type": "number"},
    },
}
PROMPT_HASH = hashlib.sha256(PROMPT_VLM.encode()).hexdigest()[:16]
VLM_RESPONSE_FORMAT = {"type": "json_schema", "json_schema": {"name": "saco_lixo_response", "schema": SCHEMA_VLM}}


In [ ]:
if client is not None:
    ok_schema = _suporta(response_format=VLM_RESPONSE_FORMAT)
    if not ok_schema:
        VLM_RESPONSE_FORMAT = None
else:
    ok_schema = True  # compatibilidade com caches existentes

print(f"Prompt hash: {PROMPT_HASH} | json_schema={VLM_RESPONSE_FORMAT is not None}")


def metadados_vlm_cache(image_path):
    image_path = Path(image_path)
    return {
        "prompt_hash": PROMPT_HASH,
        "prompt_path": str(PROMPT_CLASSIFICACAO_PATH),
        "modelo_vlm": modelo_vlm,
        "modelo_vlm_repositorio_base": MODELO_VLM_REPOSITORIO_BASE,
        "modelo_vlm_arquivo": MODELO_VLM_ARQUIVO,
        "modelo_vlm_formato": MODELO_VLM_FORMATO,
        "modelo_vlm_quantizacao": MODELO_VLM_QUANTIZACAO,
        "modelo_vlm_arquitetura": MODELO_VLM_ARQUITETURA,
        "modelo_vlm_tamanho_disco_gb": MODELO_VLM_TAMANHO_DISCO_GB,
        "lm_studio_app_version": LM_STUDIO_APP_VERSION,
        "lm_studio_base_url": LM_BASE_URL,
        "seed": SEED if VLM_SUPORTA_SEED else None,
        "temperature": 0,
        "max_tokens": VLM_CLASS_MAX_TOKENS,
        "response_format": "json_schema" if VLM_RESPONSE_FORMAT is not None else None,
        "image_id": image_path.name,
        "image_sha256": sha256_arquivo(image_path),
    }


def cache_path_vlm(image_path):
    image_path = Path(image_path)
    chave_nome = hashlib.sha256(image_path.name.encode("utf-8")).hexdigest()[:12]
    return VLM_CACHE_DIR / f"{image_path.stem}_{chave_nome}.json"


In [ ]:
def inferir_vlm(image_path: str) -> dict:
    if client is None:
        raise RuntimeError("LM Studio nao esta conectado para inferencia VLM.")
    image_path = Path(image_path)
    t_prep0 = time.perf_counter()
    imagem_payload = imagem_para_data_url_jpeg(image_path)
    cache_meta = metadados_vlm_cache(image_path)
    t_prep = (time.perf_counter() - t_prep0) * 1000

    kwargs = dict(
        model=modelo_vlm,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": PROMPT_VLM},
            {"type": "image_url", "image_url": {"url": imagem_payload["data_url"]}},
        ]}],
        max_tokens=VLM_CLASS_MAX_TOKENS,
        temperature=0,
        timeout=VLM_TIMEOUT_CLASS,
    )
    if VLM_SUPORTA_SEED:
        kwargs["seed"] = SEED
    if VLM_RESPONSE_FORMAT is not None:
        kwargs["response_format"] = VLM_RESPONSE_FORMAT

    ultimo_erro = None
    for tentativa in range(VLM_MAX_TENTATIVAS):
        try:
            t0 = time.perf_counter()
            resp = client.chat.completions.create(**kwargs)
            t_infer = (time.perf_counter() - t0) * 1000
            raw = resp.choices[0].message.content
            parsed = parse_json_robusto(raw)
            if not isinstance(parsed, dict):
                parsed = {"parse_error": True}
            if jsonschema is None:
                schema_ok = None
            else:
                try:
                    jsonschema.validate(parsed, SCHEMA_VLM)
                    schema_ok = True
                except jsonschema.ValidationError:
                    schema_ok = False
            return {
                "tem_lixo": bool(parsed.get("tem_lixo", False)),
                "descricao_breve": parsed.get("descricao_breve", ""),
                "confianca": parsed.get("confianca", 0.0),
                "parse_error": bool(parsed.get("parse_error", False)),
                "schema_valido": schema_ok,
                "raw_response": raw,
                **cache_meta,
                "image_mime_type": imagem_payload["mime_type"],
                "image_payload_bytes": imagem_payload["payload_bytes"],
                "image_payload_sha256": imagem_payload["payload_sha256"],
                "latencia_inferencia_ms": round(t_infer, 2),
                "latencia_end_to_end_ms": round(t_prep + t_infer, 2),
            }
        except Exception as e:
            ultimo_erro = str(e)
            time.sleep(2 ** tentativa)
    return {"tem_lixo": False, "parse_error": True, "erro": ultimo_erro, **cache_meta}


In [ ]:
def carregar_vlm_preds():
    agregado = OUTPUT_VLM_DIR / "predictions_vlm.json"
    if test_images:
        preds = []
        for img in test_images:
            p = cache_path_vlm(img)
            if cache_valido(p, metadados_vlm_cache(img)):
                preds.append(ler_json(p))
        return preds
    data = ler_json(agregado)
    return data if isinstance(data, list) else []


VLM_CLASS_FINGERPRINT = pipeline_fingerprint({
    "prompt_hash": PROMPT_HASH, "model": MODELO_VLM_ALVO, "test_dataset": TEST_DATASET_CONFIG,
    "temperature": 0, "seed": SEED, "max_tokens": VLM_CLASS_MAX_TOKENS,
})

pendentes_vlm = [img for img in test_images if not cache_valido(cache_path_vlm(img), metadados_vlm_cache(img))]
print(f"VLM classificacao pendente: {len(pendentes_vlm)}/{len(test_images)} imagens locais.")

if RUN_VLM_CLASSIFICACAO and pendentes_vlm:
    if client is None:
        raise RuntimeError("LM Studio precisa estar online para processar as imagens VLM pendentes.")
    concluidas = len(test_images) - len(pendentes_vlm)
    marcar_etapa("vlm_classification", "running", VLM_CLASS_FINGERPRINT, completed=concluidas, total=len(test_images))
    try:
        for idx, img_path in enumerate(tqdm(pendentes_vlm, desc="VLM classificacao"), start=1):
            res = inferir_vlm(str(img_path))
            res["image_id"] = img_path.name
            salvar_json(res, cache_path_vlm(img_path))
            if idx % 5 == 0 or idx == len(pendentes_vlm):
                marcar_etapa("vlm_classification", "running", VLM_CLASS_FINGERPRINT,
                             completed=concluidas + idx, total=len(test_images))
    except BaseException as exc:
        marcar_etapa("vlm_classification", "failed", VLM_CLASS_FINGERPRINT,
                     completed=len(carregar_vlm_preds()), total=len(test_images), error=repr(exc))
        raise

vlm_preds = carregar_vlm_preds()
vlm_by_id = {p["image_id"]: p for p in vlm_preds if isinstance(p, dict) and "image_id" in p}
if test_images and len(vlm_by_id) == len(test_images):
    salvar_json(list(vlm_by_id.values()), OUTPUT_VLM_DIR / "predictions_vlm.json")
    marcar_etapa("vlm_classification", "completed", VLM_CLASS_FINGERPRINT,
                 completed=len(vlm_by_id), total=len(test_images))

lat_v_infer = [p.get("latencia_inferencia_ms", 0) for p in vlm_preds if isinstance(p, dict) and not p.get("erro") and p.get("latencia_inferencia_ms")]
lat_v_e2e = [p.get("latencia_end_to_end_ms", 0) for p in vlm_preds if isinstance(p, dict) and not p.get("erro") and p.get("latencia_end_to_end_ms")]
n_erros_vlm = sum(1 for p in vlm_preds if isinstance(p, dict) and (p.get("parse_error") or p.get("erro")))
print(f"VLM classificacao carregado: {len(vlm_by_id)}/{len(test_image_ids)} IDs do GT | erros={n_erros_vlm}")


## 7. VLM localização: diagnóstico, execução pendente e cache


In [ ]:
if not PROMPT_LOCALIZACAO_PATH.exists():
    raise FileNotFoundError(f"Prompt de localizacao ausente: {PROMPT_LOCALIZACAO_PATH}")
PROMPT_VLM_LOCALIZACAO = PROMPT_LOCALIZACAO_PATH.read_text(encoding="utf-8").strip()

SCHEMA_VLM_LOCALIZACAO = {
    "type": "object", "required": ["tem_lixo", "deteccoes"],
    "properties": {
        "tem_lixo": {"type": "boolean"},
        "deteccoes": {"type": "array", "maxItems": VLM_LOC_MAX_DETECCOES, "items": {
            "type": "object", "required": ["label", "box_2d"],
            "properties": {
                "label": {"type": "string"},
                "box_2d": {"type": "array", "items": {"type": "number"}, "minItems": 4, "maxItems": 4},
            }}},
        "descricao_breve": {"type": "string"},
    },
}
PROMPT_LOC_HASH = hashlib.sha256(PROMPT_VLM_LOCALIZACAO.encode()).hexdigest()[:16]
VLM_LOC_RESPONSE_FORMAT = {"type": "json_schema", "json_schema": {"name": "saco_lixo_localizacao_response", "schema": SCHEMA_VLM_LOCALIZACAO}}


In [ ]:
if client is not None:
    ok_schema_loc = _suporta(response_format=VLM_LOC_RESPONSE_FORMAT)
    if not ok_schema_loc:
        VLM_LOC_RESPONSE_FORMAT = None
else:
    ok_schema_loc = True

print(f"Prompt localizacao hash: {PROMPT_LOC_HASH} | json_schema={VLM_LOC_RESPONSE_FORMAT is not None}")


def metadados_vlm_loc_cache(image_path):
    image_path = Path(image_path)
    return {
        "prompt_hash": PROMPT_LOC_HASH,
        "prompt_path": str(PROMPT_LOCALIZACAO_PATH),
        "max_deteccoes": VLM_LOC_MAX_DETECCOES,
        "modelo_vlm": modelo_vlm,
        "modelo_vlm_repositorio_base": MODELO_VLM_REPOSITORIO_BASE,
        "modelo_vlm_arquivo": MODELO_VLM_ARQUIVO,
        "modelo_vlm_formato": MODELO_VLM_FORMATO,
        "modelo_vlm_quantizacao": MODELO_VLM_QUANTIZACAO,
        "modelo_vlm_arquitetura": MODELO_VLM_ARQUITETURA,
        "modelo_vlm_tamanho_disco_gb": MODELO_VLM_TAMANHO_DISCO_GB,
        "lm_studio_app_version": LM_STUDIO_APP_VERSION,
        "lm_studio_base_url": LM_BASE_URL,
        "seed": SEED if VLM_SUPORTA_SEED else None,
        "temperature": 0,
        "max_tokens": VLM_LOC_MAX_TOKENS,
        "response_format": "json_schema" if VLM_LOC_RESPONSE_FORMAT is not None else None,
        "image_id": image_path.name,
        "image_sha256": sha256_arquivo(image_path),
    }


def cache_path_vlm_loc(image_path):
    image_path = Path(image_path)
    chave_nome = hashlib.sha256(image_path.name.encode("utf-8")).hexdigest()[:12]
    return VLM_LOC_CACHE_DIR / f"{image_path.stem}_{chave_nome}.json"


In [ ]:
def _box_1000(x_min, y_min, x_max, y_max, label="garbage bag", confidence=None):
    x1, x2 = sorted([float(x_min), float(x_max)])
    y1, y2 = sorted([float(y_min), float(y_max)])
    x1, y1 = max(0, min(1000, x1)), max(0, min(1000, y1))
    x2, y2 = max(0, min(1000, x2)), max(0, min(1000, y2))
    if x2 <= x1 or y2 <= y1:
        return None
    box = {"label": str(label or "garbage bag"), "box_2d": [round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)],
           "x_min": x1, "y_min": y1, "x_max": x2, "y_max": y2}
    if confidence is not None:
        try:
            box["confidence"] = float(confidence)
        except (TypeError, ValueError):
            pass
    return box


def normalizar_box_1000(box):
    if not isinstance(box, dict):
        return None
    label = box.get("label", box.get("nome", box.get("descricao", "garbage bag")))
    confidence = box.get("confidence", box.get("confianca"))
    raw = box.get("box_2d")
    vals = None
    if isinstance(raw, str):
        raw = [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", raw)[:4]]
    if isinstance(raw, (list, tuple)) and len(raw) >= 4:
        vals = dict(x_min=float(raw[0]), y_min=float(raw[1]), x_max=float(raw[2]), y_max=float(raw[3]))
    if vals is None:
        aliases = {"x_min": ["x_min", "xmin", "left", "x1"], "y_min": ["y_min", "ymin", "top", "y1"],
                   "x_max": ["x_max", "xmax", "right", "x2"], "y_max": ["y_max", "ymax", "bottom", "y2"]}
        vals = {canon: float(box[nome]) for canon, nomes in aliases.items() for nome in nomes if nome in box}
        if len(vals) != 4:
            return None
    if max(vals.values()) <= 1.5:
        vals = {k: v * 1000 for k, v in vals.items()}
    return _box_1000(vals["x_min"], vals["y_min"], vals["x_max"], vals["y_max"], label, confidence)


In [ ]:
def inferir_vlm_localizacao(image_path: str) -> dict:
    if client is None:
        raise RuntimeError("LM Studio nao esta conectado para inferencia VLM de localizacao.")
    image_path = Path(image_path)
    t_prep0 = time.perf_counter()
    imagem_payload = imagem_para_data_url_jpeg(image_path)
    cache_meta = metadados_vlm_loc_cache(image_path)
    t_prep = (time.perf_counter() - t_prep0) * 1000

    kwargs = dict(
        model=modelo_vlm,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": PROMPT_VLM_LOCALIZACAO},
            {"type": "image_url", "image_url": {"url": imagem_payload["data_url"]}},
        ]}],
        max_tokens=VLM_LOC_MAX_TOKENS,
        temperature=0,
        timeout=VLM_TIMEOUT_LOC,
    )
    if VLM_SUPORTA_SEED:
        kwargs["seed"] = SEED
    if VLM_LOC_RESPONSE_FORMAT is not None:
        kwargs["response_format"] = VLM_LOC_RESPONSE_FORMAT

    ultimo_erro = None
    for tentativa in range(VLM_MAX_TENTATIVAS):
        try:
            t0 = time.perf_counter()
            resp = client.chat.completions.create(**kwargs)
            t_infer = (time.perf_counter() - t0) * 1000
            raw = resp.choices[0].message.content
            parsed = parse_json_robusto(raw)
            if isinstance(parsed, list):
                parsed = {"tem_lixo": bool(parsed), "deteccoes": parsed}
            elif not isinstance(parsed, dict):
                parsed = {"parse_error": True, "deteccoes": []}
            deteccoes_raw = parsed.get("deteccoes", parsed.get("boxes", []))
            if isinstance(deteccoes_raw, dict):
                deteccoes_raw = [deteccoes_raw]
            if not isinstance(deteccoes_raw, list):
                deteccoes_raw = []
            n_deteccoes_recebidas = len(deteccoes_raw)
            deteccoes_raw = deteccoes_raw[:VLM_LOC_MAX_DETECCOES]
            parsed["deteccoes"] = deteccoes_raw
            boxes = [b for b in (normalizar_box_1000(x) for x in deteccoes_raw) if b is not None]
            if jsonschema is None:
                schema_ok = None
            else:
                try:
                    jsonschema.validate(parsed, SCHEMA_VLM_LOCALIZACAO)
                    schema_ok = True
                except jsonschema.ValidationError:
                    schema_ok = False
            return {"tem_lixo": bool(parsed.get("tem_lixo", bool(boxes))), "deteccoes": boxes, "boxes": boxes,
                    "n_boxes": len(boxes), "n_deteccoes_recebidas": n_deteccoes_recebidas,
                    "deteccoes_descartadas_limite": max(0, n_deteccoes_recebidas - VLM_LOC_MAX_DETECCOES),
                    "descricao_breve": parsed.get("descricao_breve", ""),
                    "parse_error": bool(parsed.get("parse_error", False)), "schema_valido": schema_ok,
                    "raw_response": raw, **cache_meta,
                    "latencia_inferencia_ms": round(t_infer, 2),
                    "latencia_end_to_end_ms": round(t_prep + t_infer, 2)}
        except Exception as e:
            ultimo_erro = str(e)
            time.sleep(2 ** tentativa)
    return {"tem_lixo": False, "deteccoes": [], "boxes": [], "n_boxes": 0, "parse_error": True,
            "erro": ultimo_erro, **cache_meta, "latencia_inferencia_ms": 0.0, "latencia_end_to_end_ms": 0.0}


In [ ]:
def carregar_vlm_loc_preds():
    agregado = OUTPUT_VLM_DIR / "predictions_vlm_localizacao.json"
    if test_images:
        preds = []
        for img in test_images:
            p = cache_path_vlm_loc(img)
            if cache_valido(p, metadados_vlm_loc_cache(img)):
                preds.append(ler_json(p))
        return preds
    data = ler_json(agregado)
    return data if isinstance(data, list) else []


VLM_LOC_FINGERPRINT = pipeline_fingerprint({
    "prompt_hash": PROMPT_LOC_HASH, "model": MODELO_VLM_ALVO, "test_dataset": TEST_DATASET_CONFIG,
    "max_deteccoes": VLM_LOC_MAX_DETECCOES, "temperature": 0, "seed": SEED,
    "max_tokens": VLM_LOC_MAX_TOKENS, "strategy": VLM_LOC_ESTRATEGIA,
})


def resultado_loc_vazio_por_cascata(img_path):
    meta = metadados_vlm_loc_cache(img_path)
    return {"tem_lixo": False, "deteccoes": [], "boxes": [], "n_boxes": 0,
            "descricao_breve": "sem lixo pela classificacao VLM", "parse_error": False,
            "schema_valido": True, "fonte": "cascata_classificacao_vlm", **meta,
            "latencia_inferencia_ms": 0.0, "latencia_end_to_end_ms": 0.0}

imgs_loc = list(test_images)
pendentes_loc = [img for img in imgs_loc if not cache_valido(cache_path_vlm_loc(img), metadados_vlm_loc_cache(img))]
print(f"Localizacao VLM: {len(pendentes_loc)} pendentes de {len(imgs_loc)} imagens locais.")
print(f"Arquivos no cache: {contar_arquivos_json(VLM_LOC_CACHE_DIR)} | dir={VLM_LOC_CACHE_DIR}")


In [ ]:
if RUN_VLM_LOCALIZACAO and pendentes_loc:
    if VLM_LOC_ESTRATEGIA not in {"todos", "positivos_vlm"}:
        raise ValueError("VLM_LOC_ESTRATEGIA deve ser 'todos' ou 'positivos_vlm'.")
    if client is None:
        raise RuntimeError("LM Studio precisa estar online para processar as localizacoes VLM pendentes.")
    a_processar = pendentes_loc
    if VLM_LOC_MAX_IMAGES is not None:
        a_processar = a_processar[:int(VLM_LOC_MAX_IMAGES)]
    concluidas = len(test_images) - len(pendentes_loc)
    marcar_etapa("vlm_localization", "running", VLM_LOC_FINGERPRINT, completed=concluidas, total=len(test_images))
    try:
        for idx, img_path in enumerate(tqdm(a_processar, desc="VLM localizacao"), start=1):
            pred_class = vlm_by_id.get(img_path.name, {})
            if VLM_LOC_ESTRATEGIA == "positivos_vlm" and pred_class and not pred_class.get("tem_lixo", False):
                res = resultado_loc_vazio_por_cascata(img_path)
            else:
                res = inferir_vlm_localizacao(str(img_path))
            res["image_id"] = img_path.name
            salvar_json(res, cache_path_vlm_loc(img_path))
            if idx % 5 == 0 or idx == len(a_processar):
                marcar_etapa("vlm_localization", "running", VLM_LOC_FINGERPRINT,
                             completed=concluidas + idx, total=len(test_images))
    except BaseException as exc:
        marcar_etapa("vlm_localization", "failed", VLM_LOC_FINGERPRINT,
                     completed=len(carregar_vlm_loc_preds()), total=len(test_images), error=repr(exc))
        raise
elif pendentes_loc:
    print("Inferencia nova desligada. Defina RUN_VLM_LOCALIZACAO=True para processar pendentes.")

vlm_loc_preds = carregar_vlm_loc_preds()
vlm_loc_by_id = {p["image_id"]: p for p in vlm_loc_preds if isinstance(p, dict) and "image_id" in p}
print(f"Localizacao VLM carregada: {len(vlm_loc_by_id)}/{len(test_image_ids)} IDs do GT")

if imgs_loc and len(vlm_loc_by_id) == len(imgs_loc):
    salvar_json([vlm_loc_by_id[img.name] for img in imgs_loc], OUTPUT_VLM_DIR / "predictions_vlm_localizacao.json")
    marcar_etapa("vlm_localization", "completed", VLM_LOC_FINGERPRINT,
                 completed=len(vlm_loc_by_id), total=len(test_images))
    print(f"Agregado salvo em {OUTPUT_VLM_DIR / 'predictions_vlm_localizacao.json'}")


## 8. Avaliação de localização


In [ ]:
LIMIAR_ESCOLHIDO = globals().get("LIMIAR_ESCOLHIDO", 0.25)
IOU_LOCALIZACAO = 0.50


def gt_boxes_1000(image_id):
    lbl = TESTE_LBL / (Path(image_id).stem + ".txt")
    boxes = []
    if not lbl.exists():
        return boxes
    for linha in lbl.read_text().strip().splitlines():
        parts = linha.strip().split()
        if len(parts) < 5:
            continue
        _, xc, yc, w, h = parts[:5]
        xc, yc, w, h = map(float, [xc, yc, w, h])
        box = _box_1000((xc - w/2) * 1000, (yc - h/2) * 1000, (xc + w/2) * 1000, (yc + h/2) * 1000)
        if box is not None:
            boxes.append(box)
    return boxes


def yolo_boxes_1000(image_id, limiar=LIMIAR_ESCOLHIDO):
    pred = pred_by_id.get(image_id, {})
    boxes = []
    for det in pred.get("deteccoes", []):
        if det.get("confidence", 0.0) < limiar:
            continue
        xc, yc, w, h = map(float, det["bbox_xywhn"][:4])
        box = _box_1000((xc - w/2) * 1000, (yc - h/2) * 1000, (xc + w/2) * 1000, (yc + h/2) * 1000,
                        label="lixo", confidence=det.get("confidence"))
        if box is not None:
            boxes.append(box)
    return boxes


def vlm_boxes_1000(pred):
    raw = pred.get("deteccoes", pred.get("boxes", []))
    return [b for b in (normalizar_box_1000(x) for x in raw) if b is not None]


In [ ]:
def iou_1000(a, b):
    ix1, iy1 = max(a["x_min"], b["x_min"]), max(a["y_min"], b["y_min"])
    ix2, iy2 = min(a["x_max"], b["x_max"]), min(a["y_max"], b["y_max"])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0, a["x_max"] - a["x_min"]) * max(0, a["y_max"] - a["y_min"])
    area_b = max(0, b["x_max"] - b["x_min"]) * max(0, b["y_max"] - b["y_min"])
    denom = area_a + area_b - inter
    return inter / denom if denom > 0 else 0.0


def parear_caixas_iou(pred_boxes, gt_boxes, iou_thr=IOU_LOCALIZACAO):
    candidatos = [(iou_1000(pb, gb), ip, ig) for ip, pb in enumerate(pred_boxes) for ig, gb in enumerate(gt_boxes)]
    usados_pred, usados_gt, ious_match = set(), set(), []
    for iou, ip, ig in sorted(candidatos, reverse=True):
        if iou < iou_thr or ip in usados_pred or ig in usados_gt:
            continue
        usados_pred.add(ip)
        usados_gt.add(ig)
        ious_match.append(iou)
    tp = len(ious_match)
    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp
    melhor_iou = max([c[0] for c in candidatos] or [0.0])
    return {"tp": tp, "fp": fp, "fn": fn, "melhor_iou": melhor_iou,
            "ious_match": ious_match, "sucesso_iou_05": int(tp > 0)}


def _div0(num, den):
    return round(float(num / den), 4) if den else 0.0


In [ ]:
def avaliar_localizacao(nome_modelo, pred_boxes_by_id, ids_avaliacao, extra=None):
    linhas = []
    for image_id in ids_avaliacao:
        gts = gt_boxes_1000(image_id)
        preds = pred_boxes_by_id.get(image_id, [])
        match = parear_caixas_iou(preds, gts)
        linhas.append({"image_id": image_id, "modelo": nome_modelo, "n_gt": len(gts), "n_pred": len(preds),
                       "tp_boxes": match["tp"], "fp_boxes": match["fp"], "fn_boxes": match["fn"],
                       "melhor_iou": round(match["melhor_iou"], 4),
                       "iou_medio_matches": round(float(np.mean(match["ious_match"])), 4) if match["ious_match"] else 0.0,
                       "sucesso_iou_05": match["sucesso_iou_05"]})
    df = pd.DataFrame(linhas)
    positivos = df[df["n_gt"] > 0]
    negativos = df[df["n_gt"] == 0]
    tp, fp, fn = int(df["tp_boxes"].sum()), int(df["fp_boxes"].sum()), int(df["fn_boxes"].sum())
    precisao = _div0(tp, tp + fp)
    recall = _div0(tp, tp + fn)
    f1 = _div0(2 * precisao * recall, precisao + recall)
    resumo = {"modelo": nome_modelo, "iou_threshold": IOU_LOCALIZACAO, "imagens_avaliadas": int(len(df)),
              "imagens_positivas_gt": int(len(positivos)), "imagens_negativas_gt": int(len(negativos)),
              "gt_boxes": int(df["n_gt"].sum()), "pred_boxes": int(df["n_pred"].sum()),
              "tp_boxes_iou_05": tp, "fp_boxes_iou_05": fp, "fn_boxes_iou_05": fn,
              "precisao_box_iou_05": precisao, "recall_box_iou_05": recall, "f1_box_iou_05": f1,
              "imagens_positivas_com_iou_05": int(positivos["sucesso_iou_05"].sum()) if len(positivos) else 0,
              "taxa_imagens_positivas_com_iou_05": round(float(positivos["sucesso_iou_05"].mean()), 4) if len(positivos) else 0.0,
              "iou_medio_melhor_por_imagem_positiva": round(float(positivos["melhor_iou"].mean()), 4) if len(positivos) else 0.0,
              "taxa_imagens_negativas_com_predicao": round(float((negativos["n_pred"] > 0).mean()), 4) if len(negativos) else 0.0}
    resumo["sucessos_iou_05"] = resumo["imagens_positivas_com_iou_05"]
    resumo["taxa_sucesso_iou_05"] = resumo["taxa_imagens_positivas_com_iou_05"]
    resumo["iou_medio"] = resumo["iou_medio_melhor_por_imagem_positiva"]
    if extra:
        resumo.update(extra)
    return df, resumo


In [ ]:
ids_loc_yolo = [image_id for image_id in test_image_ids if image_id in pred_by_id]
ids_loc_vlm = [image_id for image_id in test_image_ids if image_id in vlm_loc_by_id]
ids_loc = [image_id for image_id in test_image_ids if image_id in pred_by_id and image_id in vlm_loc_by_id]

if len(ids_loc) < len(test_image_ids):
    print(f"Avaliacao de localizacao parcial: {len(ids_loc)}/{len(test_image_ids)} IDs com YOLO e VLM.")

pred_boxes_yolo_by_id = {image_id: yolo_boxes_1000(image_id, LIMIAR_ESCOLHIDO) for image_id in ids_loc}
pred_boxes_vlm_by_id = {image_id: vlm_boxes_1000(vlm_loc_by_id[image_id]) for image_id in ids_loc}

if ids_loc:
    df_loc_yolo, loc_yolo_resumo = avaliar_localizacao("YOLOv11m", pred_boxes_yolo_by_id, ids_loc,
                                                       extra={"limiar_conf": LIMIAR_ESCOLHIDO})
    df_loc_vlm, loc_vlm_resumo = avaliar_localizacao(MODELO_VLM_LABEL, pred_boxes_vlm_by_id, ids_loc,
                                                     extra={"prompt_hash": PROMPT_LOC_HASH, "modelo_id": modelo_vlm})
    df_loc_yolo.to_csv("outputs/tabela_localizacao_yolo.csv", index=False)
    df_loc_vlm.to_csv(OUTPUT_VLM_DIR / "tabela_localizacao_vlm.csv", index=False)
    df_loc_comparativo = pd.DataFrame([loc_yolo_resumo, loc_vlm_resumo])
    df_loc_comparativo.to_csv(OUTPUT_VLM_DIR / "tabela_localizacao_comparativa.csv", index=False)
    salvar_json({"yolo": loc_yolo_resumo, "vlm": loc_vlm_resumo}, OUTPUT_VLM_DIR / "localizacao_comparativa.json")
    salvar_json(loc_vlm_resumo, OUTPUT_VLM_DIR / "localizacao_vlm.json")
    print(df_loc_comparativo[["modelo", "precisao_box_iou_05", "recall_box_iou_05", "f1_box_iou_05", "imagens_avaliadas"]].to_string(index=False))
else:
    df_loc_yolo = pd.DataFrame()
    df_loc_vlm = pd.DataFrame()
    loc_yolo_resumo = {}
    loc_vlm_resumo = {}
    print("Sem dados suficientes para avaliacao de localizacao.")


## 9. Métricas binárias


In [ ]:
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef


def wilson_ic(sucessos, n, z=1.96):
    if n == 0:
        return (None, None)
    p = sucessos / n
    denom = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / denom
    margem = z * ((p * (1 - p) + z**2 / (4 * n)) / n) ** 0.5 / denom
    return (round(float(max(0, centro - margem)), 4), round(float(min(1, centro + margem)), 4))


def metricas_binarias(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    acc = accuracy_score(y_true, y_pred)
    acc_inf, acc_sup = wilson_ic(int(tp + tn), int(tp + tn + fp + fn))
    return {"acuracia": round(float(acc), 4),
            "acuracia_ic95_inf": acc_inf,
            "acuracia_ic95_sup": acc_sup,
            "precisao": round(precision_score(y_true, y_pred, zero_division=0), 4),
            "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
            "especificidade": round(tn / (tn + fp), 4) if (tn + fp) else 0,
            "balanced_accuracy": round(float(balanced_accuracy_score(y_true, y_pred)), 4),
            "mcc": round(float(matthews_corrcoef(y_true, y_pred)), 4),
            "kappa": round(float(cohen_kappa_score(y_true, y_pred)), 4),
            "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
            "taxa_fp": round(fp / (fp + tn), 4) if (fp + tn) else 0,
            "matriz_confusao": [[int(tn), int(fp)], [int(fn), int(tp)]],
            "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}


def vlm_decisao(p):
    if p.get("parse_error") or p.get("erro"):
        return 0
    return int(bool(p.get("tem_lixo")))

LIMIAR_ESCOLHIDO = 0.25
ids_bin = [i for i in ids_ok if i in vlm_by_id]

if ids_bin:
    y_true_bin = [gt_map[i] for i in ids_bin]
    y_pred_y = [imagem_tem_lixo(pred_by_id[i]["deteccoes"], LIMIAR_ESCOLHIDO) for i in ids_bin]
    y_pred_v = [vlm_decisao(vlm_by_id[i]) for i in ids_bin]
    met_yolo = metricas_binarias(y_true_bin, y_pred_y)
    met_vlm = metricas_binarias(y_true_bin, y_pred_v)
    linhas_metricas = [
        {"modelo": "YOLOv11m", "n": len(ids_bin), **met_yolo},
        {"modelo": MODELO_VLM_LABEL, "n": len(ids_bin), **met_vlm},
    ]
    print(f"Avaliacao binaria pareada em {len(ids_bin)} imagens")
    print(pd.DataFrame(linhas_metricas)[["modelo", "n", "acuracia", "balanced_accuracy", "precisao", "recall", "f1", "mcc", "kappa", "taxa_fp"]].to_string(index=False))
else:
    y_true_bin = []
    y_pred_y = []
    y_pred_v = []
    linhas_metricas = []
    met_yolo = met_vlm = None
    print("Sem predicoes pareadas YOLO/VLM para metricas binarias.")


## 10. Análise comparativa


### 10.1 Preparação dos artefatos


In [ ]:
from IPython.display import display
from PIL import ImageDraw
import math

ANALISE_DIR = OUTPUT_VLM_DIR / "analises"
AMOSTRAS_DIR = ANALISE_DIR / "amostras"
ANALISE_DIR.mkdir(parents=True, exist_ok=True)
AMOSTRAS_DIR.mkdir(parents=True, exist_ok=True)
arquivos_analise = []


def registrar_arquivo(path):
    path = Path(path)
    arquivos_analise.append(str(path))
    return path


def salvar_e_mostrar_fig(fig, path):
    path = registrar_arquivo(path)
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)

print(f"Artefatos analiticos serao salvos em: {ANALISE_DIR}")


### 10.1.1 Auditoria pHash e vazamento


In [ ]:
PHASH_DIST_MAX = 5


def _phash_hex(path):
    import imagehash
    with Image.open(path) as im:
        rgb = ImageOps.exif_transpose(im).convert("RGB")
        return str(imagehash.phash(rgb))


def _coletar_phash(registros, desc):
    linhas = []
    for reg in tqdm(registros, desc=desc):
        path = Path(reg["path"])
        try:
            linhas.append({
                **reg,
                "path": str(path),
                "file_size": int(path.stat().st_size),
                "file_sha256": sha256_arquivo(path),
                "phash": _phash_hex(path),
                "erro": None,
            })
        except Exception as e:
            linhas.append({**reg, "path": str(path), "file_size": None, "file_sha256": None, "phash": None, "erro": repr(e)})
    return pd.DataFrame(linhas)


def _dist_hamming_hex(a, b):
    return (int(str(a), 16) ^ int(str(b), 16)).bit_count()


def _pares_proximos_phash(df_a, df_b, nome_a, nome_b, max_dist=PHASH_DIST_MAX, max_pares=200):
    colunas = [
        "grupo_a", "image_id_a", "source_a", "path_a",
        "grupo_b", "image_id_b", "source_b", "path_b",
        "distancia_phash",
    ]
    a = df_a.dropna(subset=["phash"]).copy()
    b = df_b.dropna(subset=["phash"]).copy()
    hashes_b = [(row.phash, row) for row in b.itertuples(index=False)]
    pares = []
    for row_a in tqdm(list(a.itertuples(index=False)), desc=f"pHash {nome_a} x {nome_b}"):
        for hash_b, row_b in hashes_b:
            dist = _dist_hamming_hex(row_a.phash, hash_b)
            if dist <= max_dist:
                pares.append({
                    "grupo_a": nome_a,
                    "image_id_a": row_a.image_id,
                    "source_a": getattr(row_a, "source", None),
                    "path_a": row_a.path,
                    "grupo_b": nome_b,
                    "image_id_b": row_b.image_id,
                    "source_b": getattr(row_b, "source", None),
                    "path_b": row_b.path,
                    "distancia_phash": int(dist),
                })
                if max_pares and len(pares) >= max_pares:
                    return pd.DataFrame(pares, columns=colunas)
    return pd.DataFrame(pares, columns=colunas)


def _duplicatas_exatas_phash(df):
    base = df.dropna(subset=["phash"])
    grupos = base.groupby("phash").filter(lambda g: len(g) > 1)
    if grupos.empty:
        return pd.DataFrame(columns=["grupo", "phash", "n", "image_ids", "paths"])
    linhas = []
    for (grupo, phash), g in grupos.groupby(["grupo", "phash"]):
        linhas.append({
            "grupo": grupo,
            "phash": phash,
            "n": len(g),
            "image_ids": ";".join(g["image_id"].astype(str).tolist()),
            "paths": ";".join(g["path"].astype(str).tolist()),
        })
    return pd.DataFrame(linhas).sort_values(["grupo", "n"], ascending=[True, False])


In [ ]:
if RUN_AUDITORIA_PHASH:
    manifest_path = Path("data/unified/manifest.csv")
    if not manifest_path.exists():
        raise FileNotFoundError("data/unified/manifest.csv nao encontrado para auditoria pHash.")

    df_manifest = pd.read_csv(manifest_path)
    registros_phash = []
    for row in df_manifest.itertuples(index=False):
        registros_phash.append({
            "grupo": str(row.split),
            "source": str(row.source),
            "image_id": Path(row.image).name,
            "path": row.image,
            "original_image": getattr(row, "original_image", None),
        })
    for img in test_images:
        registros_phash.append({
            "grupo": "teste",
            "source": "data/teste",
            "image_id": img.name,
            "path": str(img),
            "original_image": str(img),
        })

    hashes_path = registrar_arquivo(ANALISE_DIR / "auditoria_phash_hashes.csv")
    assinatura_esperada = {
        str(Path(reg["path"])): (
            int(Path(reg["path"]).stat().st_size),
            sha256_arquivo(Path(reg["path"])),
        )
        for reg in registros_phash
    }
    if hashes_path.exists():
        df_cache_phash = pd.read_csv(hashes_path)
        cache_valido = {"path", "file_size", "file_sha256"}.issubset(df_cache_phash.columns)
        if cache_valido:
            assinatura_cache = {
                str(row.path): (int(row.file_size), str(row.file_sha256))
                for row in df_cache_phash.itertuples(index=False)
            }
            cache_valido = assinatura_cache == assinatura_esperada and len(df_cache_phash) == len(registros_phash)
        if cache_valido:
            df_phash = df_cache_phash
            print(f"pHash carregado do cache: {hashes_path}")
        else:
            df_phash = _coletar_phash(registros_phash, "Calculando pHash")
            df_phash.to_csv(hashes_path, index=False)
    else:
        df_phash = _coletar_phash(registros_phash, "Calculando pHash")
        df_phash.to_csv(hashes_path, index=False)

    df_phash_resumo = (
        df_phash.groupby("grupo")
        .agg(n_imagens=("image_id", "count"),
             n_erros=("erro", lambda s: int(s.notna().sum())),
             n_hashes=("phash", "nunique"))
        .reset_index()
    )
    dup_counts = df_phash.dropna(subset=["phash"]).groupby(["grupo", "phash"]).size().reset_index(name="n")
    dup_resumo = dup_counts[dup_counts["n"] > 1].groupby("grupo").agg(
        n_hashes_duplicados_exatos=("phash", "count"),
        n_imagens_em_hashes_duplicados=("n", "sum"),
    ).reset_index()
    df_phash_resumo = df_phash_resumo.merge(dup_resumo, on="grupo", how="left").fillna(0)
    for col in ["n_hashes_duplicados_exatos", "n_imagens_em_hashes_duplicados"]:
        df_phash_resumo[col] = df_phash_resumo[col].astype(int)
    df_phash_resumo.to_csv(registrar_arquivo(ANALISE_DIR / "auditoria_phash_resumo.csv"), index=False)

    df_phash_dups = _duplicatas_exatas_phash(df_phash)
    df_phash_dups.to_csv(registrar_arquivo(ANALISE_DIR / "auditoria_phash_duplicatas_exatas.csv"), index=False)

    train_val_path = registrar_arquivo(ANALISE_DIR / "auditoria_phash_vazamento_train_val.csv")
    train_teste_path = registrar_arquivo(ANALISE_DIR / "auditoria_phash_vazamento_treino_teste.csv")
    if train_val_path.exists() and train_teste_path.exists() and hashes_path.exists():
        df_phash_train_val = pd.read_csv(train_val_path)
        df_phash_train_teste = pd.read_csv(train_teste_path)
        print("Pares pHash carregados do cache.")
    else:
        df_train = df_phash[df_phash["grupo"] == "train"]
        df_val = df_phash[df_phash["grupo"] == "val"]
        df_teste_phash = df_phash[df_phash["grupo"] == "teste"]
        df_phash_train_val = _pares_proximos_phash(df_train, df_val, "train", "val")
        df_phash_train_teste = _pares_proximos_phash(df_train, df_teste_phash, "train", "teste")
        df_phash_train_val.to_csv(train_val_path, index=False)
        df_phash_train_teste.to_csv(train_teste_path, index=False)
else:
    phash_paths = {
        "hashes": ANALISE_DIR / "auditoria_phash_hashes.csv",
        "resumo": ANALISE_DIR / "auditoria_phash_resumo.csv",
        "dups": ANALISE_DIR / "auditoria_phash_duplicatas_exatas.csv",
        "train_val": ANALISE_DIR / "auditoria_phash_vazamento_train_val.csv",
        "train_teste": ANALISE_DIR / "auditoria_phash_vazamento_treino_teste.csv",
    }
    for path in phash_paths.values():
        if path.exists():
            registrar_arquivo(path)
    df_phash = pd.read_csv(phash_paths["hashes"]) if phash_paths["hashes"].exists() else pd.DataFrame()
    df_phash_resumo = pd.read_csv(phash_paths["resumo"]) if phash_paths["resumo"].exists() else pd.DataFrame()
    df_phash_dups = pd.read_csv(phash_paths["dups"]) if phash_paths["dups"].exists() else pd.DataFrame()
    df_phash_train_val = pd.read_csv(phash_paths["train_val"]) if phash_paths["train_val"].exists() else pd.DataFrame()
    df_phash_train_teste = pd.read_csv(phash_paths["train_teste"]) if phash_paths["train_teste"].exists() else pd.DataFrame()
    print("Auditoria pHash em modo cache; defina RUN_AUDITORIA_PHASH=True para recalcular com imagens locais.")


In [ ]:
display(df_phash_resumo)

print(f"Duplicatas exatas por pHash: {len(df_phash_dups)} grupos")
print(f"Pares train-val com distancia pHash <= {PHASH_DIST_MAX}: {len(df_phash_train_val)}")
print(f"Pares train-teste com distancia pHash <= {PHASH_DIST_MAX}: {len(df_phash_train_teste)}")

if not df_phash_train_val.empty:
    display(df_phash_train_val.head(10))
if not df_phash_train_teste.empty:
    display(df_phash_train_teste.head(10))


### 10.2 Funções estatísticas pareadas


In [ ]:
def mcnemar_exato(y_a, y_b, y_true):
    correto_a = np.array(y_a) == np.array(y_true)
    correto_b = np.array(y_b) == np.array(y_true)
    a_certo_b_errado = int(np.sum(correto_a & ~correto_b))
    a_errado_b_certo = int(np.sum(~correto_a & correto_b))
    n_disc = a_certo_b_errado + a_errado_b_certo
    if n_disc == 0:
        p_valor = 1.0
    else:
        menor = min(a_certo_b_errado, a_errado_b_certo)
        prob = sum(math.comb(n_disc, k) for k in range(menor + 1)) / (2 ** n_disc)
        p_valor = min(1.0, 2 * prob)
    return {"a_certo_b_errado": a_certo_b_errado, "a_errado_b_certo": a_errado_b_certo,
            "discordantes": n_disc, "p_valor_exato": float(p_valor)}


def bootstrap_ci_pareado(valores_a, valores_b, func=np.mean, n_boot=4000, seed=SEED):
    a = np.asarray(valores_a, dtype=float)
    b = np.asarray(valores_b, dtype=float)
    if len(a) == 0:
        return {"diferenca": None, "ic95_inf": None, "ic95_sup": None}
    rng = np.random.default_rng(seed)
    idx_base = np.arange(len(a))
    diffs = [func(b[idx]) - func(a[idx]) for idx in (rng.choice(idx_base, size=len(a), replace=True) for _ in range(n_boot))]
    return {"diferenca": round(float(func(b) - func(a)), 4),
            "ic95_inf": round(float(np.percentile(diffs, 2.5)), 4),
            "ic95_sup": round(float(np.percentile(diffs, 97.5)), 4)}


def teste_sinais_exato(diferencas):
    dif = np.asarray(diferencas, dtype=float)
    positivos = int(np.sum(dif > 0))
    negativos = int(np.sum(dif < 0))
    n = positivos + negativos
    if n == 0:
        p_valor = 1.0
    else:
        menor = min(positivos, negativos)
        prob = sum(math.comb(n, k) for k in range(menor + 1)) / (2 ** n)
        p_valor = min(1.0, 2 * prob)
    return {"n_sem_empate": n, "vlm_maior": positivos, "yolo_maior": negativos,
            "p_valor_exato": float(p_valor)}


### 10.3 Tabela pareada de classificação


In [ ]:
def max_conf_yolo(image_id):
    dets = pred_by_id.get(image_id, {}).get("deteccoes", [])
    return round(float(max([d.get("confidence", 0.0) for d in dets] or [0.0])), 4)


def rotulo_erro(gt, pred):
    if gt == 1 and pred == 1:
        return "TP"
    if gt == 0 and pred == 0:
        return "TN"
    if gt == 0 and pred == 1:
        return "FP"
    return "FN"

if ids_bin:
    df_binario = pd.DataFrame({"image_id": ids_bin, "gt": y_true_bin, "yolo_pred": y_pred_y, "vlm_pred": y_pred_v})
    df_binario["yolo_resultado"] = [rotulo_erro(gt, pred) for gt, pred in zip(df_binario["gt"], df_binario["yolo_pred"])]
    df_binario["vlm_resultado"] = [rotulo_erro(gt, pred) for gt, pred in zip(df_binario["gt"], df_binario["vlm_pred"])]
    df_binario["yolo_correto"] = df_binario["gt"] == df_binario["yolo_pred"]
    df_binario["vlm_correto"] = df_binario["gt"] == df_binario["vlm_pred"]
    df_binario["yolo_score_max"] = df_binario["image_id"].map(max_conf_yolo)
    df_binario["vlm_confianca"] = df_binario["image_id"].map(lambda i: vlm_by_id.get(i, {}).get("confianca"))
    df_binario["vlm_descricao"] = df_binario["image_id"].map(lambda i: vlm_by_id.get(i, {}).get("descricao_breve"))
    df_binario["n_gt_boxes"] = df_binario["image_id"].map(lambda i: len(gt_boxes_1000(i)))
    df_binario["n_yolo_boxes"] = df_binario["image_id"].map(lambda i: len(pred_boxes_yolo_by_id.get(i, [])))
    df_binario["n_vlm_boxes"] = df_binario["image_id"].map(lambda i: len(pred_boxes_vlm_by_id.get(i, [])))
else:
    cache_bin = ANALISE_DIR / "tabela_classificacao_pareada.csv"
    df_binario = pd.read_csv(cache_bin) if cache_bin.exists() else pd.DataFrame()

if not df_binario.empty:
    df_binario.to_csv(registrar_arquivo(ANALISE_DIR / "tabela_classificacao_pareada.csv"), index=False)
display(df_binario.head(12))


### 10.4 Métricas e estatística de classificação


In [ ]:
if linhas_metricas:
    df_metricas_binarias = pd.DataFrame(linhas_metricas)
else:
    cache_metricas = ANALISE_DIR / "tabela_metricas_classificacao.csv"
    df_metricas_binarias = pd.read_csv(cache_metricas) if cache_metricas.exists() else pd.DataFrame()

if not df_metricas_binarias.empty:
    df_metricas_binarias.to_csv(registrar_arquivo(ANALISE_DIR / "tabela_metricas_classificacao.csv"), index=False)

if not df_binario.empty and {"yolo_pred", "vlm_pred", "gt", "yolo_correto", "vlm_correto"}.issubset(df_binario.columns):
    mc_class = mcnemar_exato(df_binario["yolo_pred"], df_binario["vlm_pred"], df_binario["gt"])
    ci_acc = bootstrap_ci_pareado(df_binario["yolo_correto"], df_binario["vlm_correto"])
    df_estat_binaria = pd.DataFrame([{"comparacao": "Gemma - YOLO", "metrica": "acuracia_pareada", **ci_acc, **mc_class}])
else:
    cache_estat = ANALISE_DIR / "tabela_estatistica_classificacao.csv"
    df_estat_binaria = pd.read_csv(cache_estat) if cache_estat.exists() else pd.DataFrame()

if not df_estat_binaria.empty:
    df_estat_binaria.to_csv(registrar_arquivo(ANALISE_DIR / "tabela_estatistica_classificacao.csv"), index=False)

cols_metricas = ["modelo", "n", "acuracia", "acuracia_ic95_inf", "acuracia_ic95_sup", "balanced_accuracy", "precisao", "recall", "especificidade", "f1", "mcc", "kappa", "taxa_fp", "tp", "fp", "fn", "tn"]
display(df_metricas_binarias[[c for c in cols_metricas if c in df_metricas_binarias.columns]] if not df_metricas_binarias.empty else df_metricas_binarias)
display(df_estat_binaria)


### 10.5 Gráficos de classificação


In [ ]:
metric_cols = ["acuracia", "precisao", "recall", "especificidade", "f1", "taxa_fp"]
if df_metricas_binarias.empty:
    print("Sem metricas de classificacao para grafico comparativo.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(metric_cols))
    largura = 0.36
    for offset, modelo in zip([-largura/2, largura/2], df_metricas_binarias["modelo"]):
        vals = [float(df_metricas_binarias.loc[df_metricas_binarias["modelo"] == modelo, col].iloc[0]) for col in metric_cols]
        ax.bar(x + offset, vals, width=largura, label=modelo)
    ax.set_xticks(x)
    ax.set_xticklabels([c.replace("_", " ") for c in metric_cols], rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("valor")
    ax.set_title("Classificacao binaria: metricas comparativas")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    salvar_e_mostrar_fig(fig, FIG_DIR / "fig_classificacao_metricas_comparativas.png")


In [ ]:
def matriz_confusao_2x2(met):
    return np.array(met["matriz_confusao"], dtype=int)

if met_yolo is None or met_vlm is None:
    print("Sem metricas em memoria para matrizes de confusao.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for ax, met, modelo in zip(axes, [met_yolo, met_vlm], ["YOLOv11m", MODELO_VLM_LABEL]):
        cm = matriz_confusao_2x2(met)
        ax.imshow(cm, cmap="Blues")
        ax.set_title(modelo)
        ax.set_xticks([0, 1], labels=["pred 0", "pred 1"])
        ax.set_yticks([0, 1], labels=["GT 0", "GT 1"])
        for y in range(2):
            for x_i in range(2):
                ax.text(x_i, y, int(cm[y, x_i]), ha="center", va="center", fontsize=12, color="black")
    fig.suptitle("Matrizes de confusao - classificacao binaria")
    salvar_e_mostrar_fig(fig, FIG_DIR / "fig_matrizes_confusao_classificacao.png")


In [ ]:
if df_binario.empty:
    print("Sem tabela binaria para grafico de sobreposicao de erros.")
else:
    categorias = pd.Series(np.select(
        [df_binario["yolo_correto"] & df_binario["vlm_correto"],
         df_binario["yolo_correto"] & ~df_binario["vlm_correto"],
         ~df_binario["yolo_correto"] & df_binario["vlm_correto"],
         ~df_binario["yolo_correto"] & ~df_binario["vlm_correto"]],
        ["ambos corretos", "so YOLO correto", "so Gemma correto", "ambos errados"],
        default="indefinido"
    ))
    contagem = categorias.value_counts().reindex(["ambos corretos", "so YOLO correto", "so Gemma correto", "ambos errados"], fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(contagem.index, contagem.values, color=["#4c78a8", "#f58518", "#54a24b", "#e45756"])
    ax.set_ylabel("imagens")
    ax.set_title("Sobreposicao de acertos e erros")
    ax.tick_params(axis="x", rotation=20)
    for i, v in enumerate(contagem.values):
        ax.text(i, v, str(int(v)), ha="center", va="bottom")
    salvar_e_mostrar_fig(fig, FIG_DIR / "fig_sobreposicao_erros_classificacao.png")


### 10.5.1 Trade-off qualidade x latência


In [ ]:
lat_yolo_ms = round(float(np.mean(lat_y_e2e)), 2) if lat_y_e2e else None
lat_vlm_ms = round(float(np.mean(lat_v_e2e)), 2) if lat_v_e2e else None

tradeoff_rows = []
if met_yolo is not None:
    tradeoff_rows.append({"modelo": "YOLOv11m", "f1_classificacao": met_yolo.get("f1"), "latencia_end_to_end_ms": lat_yolo_ms})
if met_vlm is not None:
    tradeoff_rows.append({"modelo": MODELO_VLM_LABEL, "f1_classificacao": met_vlm.get("f1"), "latencia_end_to_end_ms": lat_vlm_ms})

df_tradeoff = pd.DataFrame(tradeoff_rows)
if not df_tradeoff.empty:
    df_tradeoff.to_csv(registrar_arquivo(ANALISE_DIR / "tabela_tradeoff_f1_latencia.csv"), index=False)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for row in df_tradeoff.to_dict("records"):
        ax.scatter(row["latencia_end_to_end_ms"], row["f1_classificacao"], s=120)
        ax.annotate(row["modelo"], (row["latencia_end_to_end_ms"], row["f1_classificacao"]), xytext=(8, 6), textcoords="offset points")
    ax.set_xscale("log")
    ax.set_xlabel("latencia end-to-end media por imagem (ms, escala log)")
    ax.set_ylabel("F1 de classificacao")
    ax.set_ylim(0, 1.05)
    ax.set_title("Trade-off entre qualidade e latencia")
    ax.grid(True, alpha=0.25)
    salvar_e_mostrar_fig(fig, FIG_DIR / "fig_tradeoff_f1_latencia.png")
else:
    print("Sem dados de F1/latencia para grafico de trade-off.")


### 10.6 Funções para amostras visuais


In [ ]:
def img_path_por_id(image_id):
    return next((p for p in test_images if p.name == image_id), None)


def box_1000_para_px(box, width, height):
    return [int(round(box["x_min"] / 1000 * width)), int(round(box["y_min"] / 1000 * height)),
            int(round(box["x_max"] / 1000 * width)), int(round(box["y_max"] / 1000 * height))]


def imagem_anotada(image_id, incluir_yolo=True, incluir_vlm=True):
    path = img_path_por_id(image_id)
    if path is None:
        return None
    with Image.open(path) as im:
        canvas = ImageOps.exif_transpose(im).convert("RGB")
    draw = ImageDraw.Draw(canvas)
    w, h = canvas.size
    for box in gt_boxes_1000(image_id):
        draw.rectangle(box_1000_para_px(box, w, h), outline=(0, 180, 0), width=4)
    if incluir_yolo:
        for box in pred_boxes_yolo_by_id.get(image_id, []):
            draw.rectangle(box_1000_para_px(box, w, h), outline=(220, 40, 40), width=3)
    if incluir_vlm:
        for box in pred_boxes_vlm_by_id.get(image_id, []):
            draw.rectangle(box_1000_para_px(box, w, h), outline=(40, 110, 230), width=3)
    return canvas


In [ ]:
def plot_grade_amostras(df_amostras, titulo, path, max_itens=8):
    if df_amostras.empty:
        print(f"Sem amostras para: {titulo}")
        return None
    df_plot = df_amostras.head(max_itens).copy()
    df_plot = df_plot[df_plot["image_id"].map(lambda image_id: img_path_por_id(image_id) is not None)]
    if df_plot.empty:
        print(f"Sem imagens locais para desenhar: {titulo}")
        return None
    n = len(df_plot)
    cols = min(4, n)
    rows = int(math.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.6 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, row in zip(axes, df_plot.to_dict("records")):
        img = imagem_anotada(row["image_id"])
        if img is None:
            continue
        stem = Path(row["image_id"]).stem
        curto = stem[:32] + ("..." if len(stem) > 32 else "")
        ax.imshow(img)
        ax.set_title(
            f'{row.get("tipo_amostra", "")} | GT={row.get("gt")} YOLO={row.get("yolo_pred")} Gemma={row.get("vlm_pred")}\n{curto}',
            fontsize=7,
            pad=4,
        )
        ax.axis("off")
    fig.suptitle(titulo + " | verde=GT, vermelho=YOLO, azul=Gemma", fontsize=12)
    salvar_e_mostrar_fig(fig, path)
    return path


### 10.7 Amostras FP/FN de classificação


In [ ]:
amostras_class = []
if not df_binario.empty:
    for modelo, col in [("YOLOv11m", "yolo_resultado"), (MODELO_VLM_LABEL, "vlm_resultado")]:
        for tipo in ["FP", "FN", "TP", "TN"]:
            subset = df_binario[df_binario[col] == tipo].copy().head(8)
            if len(subset):
                subset.insert(0, "modelo_amostra", modelo)
                subset.insert(1, "tipo_amostra", tipo)
                amostras_class.append(subset)

df_amostras_class = pd.concat(amostras_class, ignore_index=True) if amostras_class else pd.DataFrame()
if not df_amostras_class.empty:
    df_amostras_class.to_csv(registrar_arquivo(ANALISE_DIR / "amostras_classificacao_fp_fn_tp_tn.csv"), index=False)
display(df_amostras_class.head(16))


In [ ]:
if not df_amostras_class.empty:
    plot_grade_amostras(
        df_amostras_class[(df_amostras_class["modelo_amostra"] == "YOLOv11m") & (df_amostras_class["tipo_amostra"].isin(["FP", "FN"]))],
        "Amostras de erros de classificacao - YOLOv11m",
        FIG_DIR / "fig_amostras_erros_classificacao_yolo.png",
    )


In [ ]:
if not df_amostras_class.empty:
    plot_grade_amostras(
        df_amostras_class[(df_amostras_class["modelo_amostra"] == MODELO_VLM_LABEL) & (df_amostras_class["tipo_amostra"].isin(["FP", "FN"]))],
        f"Amostras de erros de classificacao - {MODELO_VLM_LABEL}",
        FIG_DIR / "fig_amostras_erros_classificacao_gemma.png",
    )


### 10.8 Tabelas pareadas de localização


In [ ]:
if not df_loc_yolo.empty and not df_loc_vlm.empty:
    df_loc_pareada = df_loc_yolo.merge(df_loc_vlm, on="image_id", suffixes=("_yolo", "_vlm"))
else:
    cache_loc_pareada = ANALISE_DIR / "tabela_localizacao_pareada.csv"
    df_loc_pareada = pd.read_csv(cache_loc_pareada) if cache_loc_pareada.exists() else pd.DataFrame()

if loc_yolo_resumo and loc_vlm_resumo:
    loc_metricas = pd.DataFrame([loc_yolo_resumo, loc_vlm_resumo])
else:
    cache_loc_metricas = ANALISE_DIR / "tabela_metricas_localizacao.csv"
    loc_metricas = pd.read_csv(cache_loc_metricas) if cache_loc_metricas.exists() else pd.DataFrame()

if not df_loc_pareada.empty:
    df_loc_pareada.to_csv(registrar_arquivo(ANALISE_DIR / "tabela_localizacao_pareada.csv"), index=False)
if not loc_metricas.empty:
    loc_metricas.to_csv(registrar_arquivo(ANALISE_DIR / "tabela_metricas_localizacao.csv"), index=False)

display(loc_metricas[["modelo", "precisao_box_iou_05", "recall_box_iou_05", "f1_box_iou_05", "taxa_sucesso_iou_05", "iou_medio"]] if not loc_metricas.empty else loc_metricas)
display(df_loc_pareada.head(12))


### 10.9 Estatística de localização


In [ ]:
estat_loc = []
if not df_loc_pareada.empty and "n_gt_yolo" in df_loc_pareada.columns:
    positivo_mask = df_loc_pareada["n_gt_yolo"].to_numpy() > 0
    yolo_sucesso = df_loc_pareada["sucesso_iou_05_yolo"].astype(int).to_numpy()
    vlm_sucesso = df_loc_pareada["sucesso_iou_05_vlm"].astype(int).to_numpy()

    if positivo_mask.any():
        mc_loc = mcnemar_exato(yolo_sucesso[positivo_mask], vlm_sucesso[positivo_mask], np.ones(int(positivo_mask.sum()), dtype=int))
        ci_sucesso = bootstrap_ci_pareado(yolo_sucesso[positivo_mask], vlm_sucesso[positivo_mask])
        sinal_iou = teste_sinais_exato(
            df_loc_pareada.loc[positivo_mask, "melhor_iou_vlm"].to_numpy() - df_loc_pareada.loc[positivo_mask, "melhor_iou_yolo"].to_numpy()
        )
        ci_iou = bootstrap_ci_pareado(df_loc_pareada.loc[positivo_mask, "melhor_iou_yolo"], df_loc_pareada.loc[positivo_mask, "melhor_iou_vlm"])
        estat_loc.append({"comparacao": "Gemma - YOLO", "metrica": "taxa_sucesso_iou_05_positivos", **ci_sucesso, **mc_loc})
        estat_loc.append({"comparacao": "Gemma - YOLO", "metrica": "melhor_iou_medio_positivos", **ci_iou, **sinal_iou})

df_estat_loc = pd.DataFrame(estat_loc)
if df_estat_loc.empty:
    cache_estat_loc = ANALISE_DIR / "tabela_estatistica_localizacao.csv"
    df_estat_loc = pd.read_csv(cache_estat_loc) if cache_estat_loc.exists() else pd.DataFrame()
if not df_estat_loc.empty:
    df_estat_loc.to_csv(registrar_arquivo(ANALISE_DIR / "tabela_estatistica_localizacao.csv"), index=False)
display(df_estat_loc)


### 10.10 Gráficos de localização


In [ ]:
metric_cols_loc = ["precisao_box_iou_05", "recall_box_iou_05", "f1_box_iou_05", "taxa_sucesso_iou_05", "iou_medio"]
if loc_metricas.empty:
    print("Sem metricas de localizacao para grafico comparativo.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(metric_cols_loc))
    largura = 0.36
    for offset, modelo in zip([-largura/2, largura/2], loc_metricas["modelo"]):
        vals = [float(loc_metricas.loc[loc_metricas["modelo"] == modelo, col].iloc[0]) for col in metric_cols_loc]
        ax.bar(x + offset, vals, width=largura, label=modelo)
    ax.set_xticks(x)
    ax.set_xticklabels([c.replace("_", " ") for c in metric_cols_loc], rotation=25, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("valor")
    ax.set_title("Localizacao: metricas comparativas")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    salvar_e_mostrar_fig(fig, FIG_DIR / "fig_localizacao_metricas_comparativas.png")


In [ ]:
if df_loc_pareada.empty:
    print("Sem dados pareados de localizacao para histograma de IoU.")
else:
    positivos = df_loc_pareada[df_loc_pareada["n_gt_yolo"] > 0]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(positivos["melhor_iou_yolo"], bins=np.linspace(0, 1, 21), alpha=0.65, label="YOLOv11m")
    ax.hist(positivos["melhor_iou_vlm"], bins=np.linspace(0, 1, 21), alpha=0.65, label=MODELO_VLM_LABEL)
    ax.axvline(IOU_LOCALIZACAO, color="black", linestyle="--", linewidth=1, label=f"IoU={IOU_LOCALIZACAO}")
    ax.set_xlabel("melhor IoU por imagem positiva")
    ax.set_ylabel("imagens")
    ax.set_title("Distribuicao do melhor IoU em imagens positivas")
    ax.legend()
    salvar_e_mostrar_fig(fig, FIG_DIR / "fig_distribuicao_iou_localizacao.png")


In [ ]:
if loc_metricas.empty:
    print("Sem metricas de localizacao para contagem TP/FP/FN.")
else:
    yolo_row = loc_metricas[loc_metricas["modelo"] == "YOLOv11m"].iloc[0].to_dict()
    vlm_row = loc_metricas[loc_metricas["modelo"] == MODELO_VLM_LABEL].iloc[0].to_dict()
    cont_loc = pd.DataFrame([
        {"modelo": "YOLOv11m", "tp_boxes": yolo_row["tp_boxes_iou_05"], "fp_boxes": yolo_row["fp_boxes_iou_05"], "fn_boxes": yolo_row["fn_boxes_iou_05"]},
        {"modelo": MODELO_VLM_LABEL, "tp_boxes": vlm_row["tp_boxes_iou_05"], "fp_boxes": vlm_row["fp_boxes_iou_05"], "fn_boxes": vlm_row["fn_boxes_iou_05"]},
    ])
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bottom = np.zeros(len(cont_loc))
    cores = {"tp_boxes": "#4c78a8", "fp_boxes": "#e45756", "fn_boxes": "#f58518"}
    for col in ["tp_boxes", "fp_boxes", "fn_boxes"]:
        vals = cont_loc[col].to_numpy(dtype=float)
        ax.bar(cont_loc["modelo"], vals, bottom=bottom, label=col.replace("_", " "), color=cores[col])
        bottom += vals
    ax.set_ylabel("caixas")
    ax.set_title("Contagem de caixas TP/FP/FN em IoU >= 0.50")
    ax.legend()
    salvar_e_mostrar_fig(fig, FIG_DIR / "fig_caixas_tp_fp_fn_localizacao.png")


### 10.11 Amostras FP/FN de localização


In [ ]:
if df_loc_pareada.empty:
    df_amostras_loc = pd.DataFrame()
else:
    criterios_loc = [
        ("yolo_fp_localizacao", (df_loc_pareada["n_gt_yolo"] == 0) & (df_loc_pareada["n_pred_yolo"] > 0)),
        ("yolo_fn_localizacao", (df_loc_pareada["n_gt_yolo"] > 0) & (df_loc_pareada["fn_boxes_yolo"] > 0)),
        ("gemma_fp_localizacao", (df_loc_pareada["n_gt_vlm"] == 0) & (df_loc_pareada["n_pred_vlm"] > 0)),
        ("gemma_fn_localizacao", (df_loc_pareada["n_gt_vlm"] > 0) & (df_loc_pareada["fn_boxes_vlm"] > 0)),
        ("gemma_acerta_yolo_erra", (df_loc_pareada["sucesso_iou_05_vlm"] == 1) & (df_loc_pareada["sucesso_iou_05_yolo"] == 0)),
        ("yolo_acerta_gemma_erra", (df_loc_pareada["sucesso_iou_05_yolo"] == 1) & (df_loc_pareada["sucesso_iou_05_vlm"] == 0)),
    ]

    amostras_loc = []
    for nome, mask in criterios_loc:
        subset = df_loc_pareada[mask].copy().head(6)
        if len(subset):
            subset.insert(0, "tipo_amostra", nome)
            amostras_loc.append(subset)
    df_amostras_loc = pd.concat(amostras_loc, ignore_index=True) if amostras_loc else pd.DataFrame()

if not df_amostras_loc.empty:
    df_amostras_loc.to_csv(registrar_arquivo(ANALISE_DIR / "amostras_localizacao_fp_fn.csv"), index=False)
display(df_amostras_loc.head(18))


In [ ]:
if df_amostras_loc.empty:
    print("Sem amostras de localizacao para plotar.")
else:
    df_plot_loc = df_amostras_loc.head(8).copy()
    df_plot_loc["gt"] = df_plot_loc["n_gt_yolo"].gt(0).astype(int)
    df_plot_loc["yolo_pred"] = df_plot_loc["n_pred_yolo"].gt(0).astype(int)
    df_plot_loc["vlm_pred"] = df_plot_loc["n_pred_vlm"].gt(0).astype(int)
    plot_grade_amostras(df_plot_loc, "Amostras de localizacao e erros por caixas", FIG_DIR / "fig_amostras_localizacao_fp_fn.png")


### 10.12 Lista de artefatos gerados


In [ ]:
df_artefatos_analise = pd.DataFrame({"arquivo": arquivos_analise})
display(df_artefatos_analise)
print(f"Total de artefatos analiticos: {len(arquivos_analise)}")


## 11. Exportação final


In [ ]:
def config_datasets_roboflow():
    return {
        "treino_positivo": DATASET_CONFIG,
        "treino_negativo": {
            "source_project": "garbage-mvzg3", **BACKGROUND_CONFIG,
            "selection_rule": "Roadway sem anotacao trash bag",
        },
        "teste": {**TEST_DATASET_CONFIG, "papel": "teste_externo"},
        "fingerprints": {"train": TRAIN_DATA_CONFIG_FINGERPRINT, "test": TEST_DATA_CONFIG_FINGERPRINT},
        "prepare_script_sha256": PREPARE_SCRIPT_SHA256,
    }


resumo = {
    "projeto": f"YOLOv11m vs {MODELO_VLM_LABEL} - notebook otimizado",
    "data_execucao": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "hardware": GPU_NAME,
    "seed": SEED,
    "modelo_vlm": modelo_vlm,
    "modelo_vlm_repositorio_base": MODELO_VLM_REPOSITORIO_BASE,
    "modelo_vlm_arquivo": MODELO_VLM_ARQUIVO,
    "modelo_vlm_formato": MODELO_VLM_FORMATO,
    "modelo_vlm_quantizacao": MODELO_VLM_QUANTIZACAO,
    "modelo_vlm_arquitetura": MODELO_VLM_ARQUITETURA,
    "modelo_vlm_tamanho_disco_gb": MODELO_VLM_TAMANHO_DISCO_GB,
    "lm_studio_app_version": LM_STUDIO_APP_VERSION,
    "slug_vlm": VLM_RUN_SLUG,
    "teste": {"total": len(df_gt), "com_lixo": n_com, "sem_lixo": n_sem},
    "cache": {
        "yolo": f"{len(pred_by_id)}/{len(test_images)}",
        "vlm_classificacao": f"{len(vlm_by_id)}/{len(test_images)}",
        "vlm_localizacao": f"{len(vlm_loc_by_id)}/{len(test_images)}",
        "validacao_estrita": CACHE_VALIDACAO_ESTRITA,
        "pipeline_state": str(PIPELINE_STATE_PATH),
    },
    "prompts": {
        "classificacao": {"path": str(PROMPT_CLASSIFICACAO_PATH), "sha256_16": PROMPT_HASH},
        "localizacao": {"path": str(PROMPT_LOCALIZACAO_PATH), "sha256_16": PROMPT_LOC_HASH,
                         "max_deteccoes": VLM_LOC_MAX_DETECCOES},
    },
}
if met_yolo is not None:
    resumo["yolo"] = {"limiar_principal": LIMIAR_ESCOLHIDO, **met_yolo,
                      "latencia_inferencia_media_ms": round(float(np.mean(lat_y_infer)), 2) if lat_y_infer else None,
                      "latencia_end_to_end_media_ms": round(float(np.mean(lat_y_e2e)), 2) if lat_y_e2e else None}
if met_vlm is not None:
    resumo["vlm"] = {"prompt_hash": PROMPT_HASH, "prompt_localizacao_hash": PROMPT_LOC_HASH,
                    "respostas_invalidas": n_erros_vlm, **met_vlm,
                    "latencia_inferencia_media_ms": round(float(np.mean(lat_v_infer)), 2) if lat_v_infer else None,
                    "latencia_end_to_end_media_ms": round(float(np.mean(lat_v_e2e)), 2) if lat_v_e2e else None}
if "loc_yolo_resumo" in globals():
    resumo["yolo_localizacao"] = loc_yolo_resumo
if "loc_vlm_resumo" in globals():
    resumo["vlm_localizacao"] = loc_vlm_resumo

if "arquivos_analise" in globals():
    resumo["artefatos_analise"] = arquivos_analise

if "df_phash_resumo" in globals() and not df_phash_resumo.empty:
    resumo["auditoria_phash"] = {
        "distancia_maxima_vazamento": PHASH_DIST_MAX,
        "resumo": df_phash_resumo.to_dict(orient="records"),
        "duplicatas_exatas_grupos": int(len(df_phash_dups)),
        "pares_train_val_suspeitos": int(len(df_phash_train_val)),
        "pares_train_teste_suspeitos": int(len(df_phash_train_teste)),
    }

yolo_prov_path = TRAIN_DIR / "run_provenance.json"
yolo_proveniencia = ler_json(yolo_prov_path, default={})
if yolo_proveniencia:
    resumo["yolo_proveniencia"] = yolo_proveniencia
else:
    resumo["yolo_proveniencia"] = {
        "yolo_run_name": YOLO_RUN_NAME,
        "weights_path": str(BEST_PT),
        "weights_exists": BEST_PT.exists(),
        "weights_sha256": sha256_arquivo(BEST_PT) if BEST_PT.exists() else None,
    }

resumo["dados"] = {
    "configuracao_datasets": config_datasets_roboflow(),
    "unified_summary": ler_json("data/unified/summary.json", default={}),
    "unified_split_audit": ler_json("data/unified/split_audit.json", default={}),
    "teste_summary": ler_json("data/teste/summary.json", default={}),
    "unified_manifest": hash_manifesto_diretorio("data/unified", EXTS_MANIFESTO_DADOS),
    "teste_manifest": hash_manifesto_diretorio("data/teste", EXTS_MANIFESTO_DADOS),
}
resumo["ambiente"] = {
    "python": sys.version.split()[0],
    "pacotes": versoes_pacotes(),
    "flags_execucao": {
        "RUN_ALL_PIPELINE": RUN_ALL_PIPELINE,
        "TRAIN_ONLY_PIPELINE": TRAIN_ONLY_PIPELINE,
        "RUN_TREINO_YOLO": RUN_TREINO_YOLO,
        "RUN_YOLO_INFERENCIA": RUN_YOLO_INFERENCIA,
        "RUN_VLM_CLASSIFICACAO": RUN_VLM_CLASSIFICACAO,
        "RUN_VLM_LOCALIZACAO": RUN_VLM_LOCALIZACAO,
        "RUN_DOWNLOAD_DADOS": RUN_DOWNLOAD_DADOS,
        "RUN_DOWNLOAD_TREINO": RUN_DOWNLOAD_TREINO,
        "RUN_DOWNLOAD_TESTE": RUN_DOWNLOAD_TESTE,
        "RUN_RECONSTRUIR_UNIFIED": RUN_RECONSTRUIR_UNIFIED,
        "RUN_RECONSTRUIR_TESTE": RUN_RECONSTRUIR_TESTE,
        "RUN_AUDITORIA_PHASH": RUN_AUDITORIA_PHASH,
        "FORCE_DOWNLOAD_DADOS": FORCE_DOWNLOAD_DADOS,
        "FORCE_DOWNLOAD_TREINO": FORCE_DOWNLOAD_TREINO,
        "FORCE_DOWNLOAD_TESTE": FORCE_DOWNLOAD_TESTE,
        "FORCE_REBUILD_DADOS": FORCE_REBUILD_DADOS,
        "FORCE_TREINO_YOLO": FORCE_TREINO_YOLO,
    },
    "pipeline_steps": pipeline_state.get("steps", {}),
}

metricas_path = OUTPUT_VLM_DIR / "metricas_finais_otimizado.json"
salvar_json(resumo, metricas_path)
print(f"Resumo salvo em {metricas_path}")
